In [1]:
ENV["MPLBACKEND"] = "agg"
using QuantumOptics
using PyPlot
using LinearAlgebra
using IonSim

# -------------------------------------------------------------
# Math Helpers & Exact QCRB Eigen-Solver
# -------------------------------------------------------------
function laguerre(n::Int, x::Float64)
    if n == 0 return 1.0 end
    if n == 1 return 1.0 - x end
    L_prev2 = 1.0
    L_prev1 = 1.0 - x
    L_curr = 0.0
    for k in 1:(n-1)
        L_curr = ((2k + 1 - x) * L_prev1 - k * L_prev2) / (k + 1)
        L_prev2 = L_prev1
        L_prev1 = L_curr
    end
    return L_curr
end

function get_exact_beta_eff(beta_target::Float64, Gamma_t::Float64)
    if Gamma_t < 1e-5 return beta_target / sqrt(3.0) end
    x = Gamma_t / 2.0
    term1 = 1.0
    term2 = (2.0 / x) * (exp(-x) - 1.0)
    term3 = -(1.0 / (2.0 * x)) * (exp(-2.0 * x) - 1.0)
    avg_sq = (beta_target^2 / x^2) * (term1 + term2 + term3)
    return sqrt(max(0.0, avg_sq))
end

function calculate_qcrb(state, state_perturbed, d_alpha::Float64; tol=1e-10)
    rho_dense_data = dense(state).data
    rho_pert_dense_data = dense(state_perturbed).data
    drho_data = (rho_pert_dense_data - rho_dense_data) / d_alpha
    λ, V = LinearAlgebra.eigen(Hermitian(rho_dense_data))
    
    qfi = 0.0
    n = length(λ)
    for i in 1:n
        for j in 1:n
            if real(λ[i] + λ[j]) > tol
                @views v_i = V[:, i]
                @views v_j = V[:, j]
                drho_ij = dot(v_i, drho_data * v_j)
                qfi += 2 * (abs(drho_ij)^2) / real(λ[i] + λ[j])
            end
        end
    end
    return qfi > 0 ? 1.0 / qfi : Inf
end

# -------------------------------------------------------------
# 1. Simulation ODE Logic
# -------------------------------------------------------------
function SDF_apply_with_alpha_dephasing(laser_intensity = 1e4, angle = 0.0, target_disp = 0.716, is_SDF = false, initial_state = nothing, fock_state_n = 0, gamma_phi_set = 1e-2, baseline_beta = 1.0)
    trap_freq = 19e5 
    ion = Ca40([("S1/2", -1/2, "g"), ("D5/2", -5/2, "e")])
    chain = LinearChain(ions=[ion], comfrequencies=(x=3e6, y=3e6, z=trap_freq), selectedmodes=(; z=[1]))
    mode = zmodes(chain)[1] 
    mode.N = 50; mode.shape = [51]
    
    k_vec = (x̂ + ẑ) / √2
    ϵ_vec = ŷ
    B_field = 1e-13
    trap_pre = Chamber(iontrap=chain, B=B_field, Bhat=ẑ)
    transition_λ = transitionwavelength(ion, ("g", "e"), trap_pre)
    
    ψ_initial = initial_state !== nothing ? initial_state : (normalize(ion["g"]) ⊗ fockstate(mode, fock_state_n))

    hbar = 1.0545718e-34; amu = 1.66053906e-27; m_Ca40 = 39.96259 * amu    
    eta = ((2 * π / transition_λ) / sqrt(2)) * sqrt(hbar / (2 * m_Ca40 * 2 * π * trap_freq))
    dummy_laser = Laser(I=laser_intensity, k=k_vec, ϵ=ϵ_vec, λ=transition_λ)
    Omega_eff = IonSim.matrixelement(ion, ("g", "e"), laser_intensity, polarization(dummy_laser), wavevector(dummy_laser), (; x=0.0, y=0.0, z=1.0)) * 2 * π

   function SDF_apply_with_alpha_dephasing(laser_intensity = 1e4, angle = 0.0, target_disp = 0.716, is_SDF = false, initial_state = nothing, fock_state_n = 0, gamma_phi_set = 1e-2, gamma_cool_set = 0.0, gamma_heat_set = 0.0, baseline_beta = 1.0)
    trap_freq = 19e5 
    ion = Ca40([("S1/2", -1/2, "g"), ("D5/2", -5/2, "e")])
    chain = LinearChain(ions=[ion], comfrequencies=(x=3e6, y=3e6, z=trap_freq), selectedmodes=(; z=[1]))
    mode = zmodes(chain)[1] 
    mode.N = 50; mode.shape = [51]
    
    k_vec = (x̂ + ẑ) / √2
    ϵ_vec = ŷ
    B_field = 1e-13
    trap_pre = Chamber(iontrap=chain, B=B_field, Bhat=ẑ)
    transition_λ = transitionwavelength(ion, ("g", "e"), trap_pre)
    
    ψ_initial = initial_state !== nothing ? initial_state : (normalize(ion["g"]) ⊗ fockstate(mode, fock_state_n))

    hbar = 1.0545718e-34; amu = 1.66053906e-27; m_Ca40 = 39.96259 * amu    
    eta = ((2 * π / transition_λ) / sqrt(2)) * sqrt(hbar / (2 * m_Ca40 * 2 * π * trap_freq))
    dummy_laser = Laser(I=laser_intensity, k=k_vec, ϵ=ϵ_vec, λ=transition_λ)
    Omega_eff = IonSim.matrixelement(ion, ("g", "e"), laser_intensity, polarization(dummy_laser), wavevector(dummy_laser), (; x=0.0, y=0.0, z=1.0)) * 2 * π

    if is_SDF
        alpha_complex = target_disp * exp(1im * angle)
        g, e = ion["g"], ion["e"]
        x_plus, x_minus = normalize(g + e), normalize(g - e)
        
        P_plus, P_minus = x_plus ⊗ dagger(x_plus), x_minus ⊗ dagger(x_minus)
        D_plus, D_minus = IonSim.displace(mode, alpha_complex), IonSim.displace(mode, -alpha_complex)
        
        U_SDF = (P_plus ⊗ D_plus) + (P_minus ⊗ D_minus)
        return isa(ψ_initial, Operator) ? U_SDF * ψ_initial * dagger(U_SDF) : U_SDF * ψ_initial
    else
        I_spin = identityoperator(ion["g"].basis)  
        
        # Convert Hz to inverse microseconds for the ODE solver timeframe
        gamma_phi_us = gamma_phi_set * 1e-6 
        gamma_cool_us = gamma_cool_set * 1e-6
        gamma_heat_us = gamma_heat_set * 1e-6

        b_fock = FockBasis(mode.N) 
        a_op = DenseOperator(mode, mode, QuantumOptics.destroy(b_fock).data)
        at_op = DenseOperator(mode, mode, QuantumOptics.create(b_fock).data)
        n_op = DenseOperator(mode, mode, QuantumOptics.number(b_fock).data)

        target_time_s = (2 * baseline_beta) / (eta * Omega_eff)
        target_time_us = target_time_s * 1e6

        drive_rate = target_disp / target_time_us
        H_m = (1im * drive_rate) * (exp(1im * angle) * at_op - exp(-1im * angle) * a_op)
        H_sys = I_spin ⊗ H_m

        # Dynamically append Jump Operators (L_k) to the array
        J_ops = typeof(H_sys)[] # Typed array to prevent QuantumOptics evaluation errors
        
        if gamma_phi_set > 0
            push!(J_ops, sqrt(gamma_phi_us) * (I_spin ⊗ n_op))
        end
        if gamma_cool_set > 0
            push!(J_ops, sqrt(gamma_cool_us) * (I_spin ⊗ a_op))
        end
        if gamma_heat_set > 0
            push!(J_ops, sqrt(gamma_heat_us) * (I_spin ⊗ at_op))
        end

        f_dyn(time_t, rho_state) = (H_sys, J_ops, dagger.(J_ops))

        rho_initial = isa(ψ_initial, Ket) ? dm(ψ_initial) : ψ_initial
        _, sol_beta = timeevolution.master_dynamic([0.0, target_time_us], rho_initial, f_dyn, reltol=1e-9, abstol=1e-11)
        return sol_beta[end]
    end
end
end

# -------------------------------------------------------------
# 2. Theory: Stepwise Open System (Trotterization)
# -------------------------------------------------------------
# -------------------------------------------------------------
# Unified Stepwise Open System (Dephasing + Cooling + Heating)
# -------------------------------------------------------------
function get_stepwise_combined_theory_rho(n_fock::Int, Gamma_phi_t::Float64, Gamma_cool_t::Float64, Gamma_heat_t::Float64, beta_target::Float64, steps::Int=600)
    N_cut = 50 # Fixed to match sim_mode.N = 50 and b_theory = FockBasis(50)
    b = FockBasis(N_cut)
    rho = dm(fockstate(b, n_fock))
    
    a_op = QuantumOptics.destroy(b)
    at_op = QuantumOptics.create(b)
    n_op = QuantumOptics.number(b)
    I_op = QuantumOptics.identityoperator(b)
    
    # Differential steps
    d_beta = beta_target / steps
    d_g_phi = Gamma_phi_t / steps
    d_g_c = Gamma_cool_t / steps
    d_g_h = Gamma_heat_t / steps
    
    D_step = QuantumOptics.displace(b, d_beta)
    D_step_dag = QuantumOptics.dagger(D_step)
    
    # Kraus Operators for Cooling (Amplitude Damping)
    M0_c = I_op - (d_g_c / 2.0) * n_op
    M1_c = sqrt(d_g_c) * a_op
    
    # Kraus Operators for Heating (Thermal Excitation)
    M0_h = I_op - (d_g_h / 2.0) * (n_op + I_op)
    M1_h = sqrt(d_g_h) * at_op
    
    for k in 1:steps
        # 1. Coherent Laser Drive
        rho = D_step * rho * D_step_dag
        
        # 2. Pure Dephasing (Fast exact diagonal projection)
        if d_g_phi > 0
            for i in 1:(N_cut + 1)
                for j in 1:(N_cut + 1)
                    rho.data[i, j] *= exp(-0.5 * d_g_phi * ((i - 1) - (j - 1))^2)
                end
            end
        end
        
        # 3. Cooling Channel
        if d_g_c > 0
            rho = M0_c * rho * dagger(M0_c) + M1_c * rho * dagger(M1_c)
        end
        
        # 4. Heating Channel
        if d_g_h > 0
            rho = M0_h * rho * dagger(M0_h) + M1_h * rho * dagger(M1_h)
        end
    end
    
    return rho
end

# -------------------------------------------------------------
# 3. Theory: Static β_eff Approximation
# -------------------------------------------------------------
function get_static_open_theory_rho(n_fock::Int, Gamma_t::Float64, beta_target::Float64)
    N_cut = 50 
    b = FockBasis(N_cut)
    
    beta_eff = get_exact_beta_eff(beta_target, Gamma_t)
    
    D_eff = displace(b, beta_eff)
    rho = D_eff * dm(fockstate(b, n_fock)) * dagger(D_eff)
    
    for i in 1:(N_cut + 1)
        for j in 1:(N_cut + 1)
            rho.data[i, j] *= exp(-0.5 * Gamma_t * ((i - 1) - (j - 1))^2)
        end
    end
    return rho
end

# -------------------------------------------------------------
# Quad-Plot Main Evaluation Function
# -------------------------------------------------------------
function plot_qcrb_macroscopic_beta_step(n_fock::Int=1, laser_intensity::Float64=1e3, gamma_phi::Float64=365.0,cool::Float64=0.0, heat::Float64=0.0, beta_signal::Float64=2.0, steps::Int=10)
    d_beta = 1e-4 
    
    trap_freq = 19e5; hbar = 1.0545718e-34; amu = 1.66053906e-27; m_Ca40 = 39.96259 * amu    
    ion = Ca40([("S1/2", -1/2, "g"), ("D5/2", -5/2, "e")])
    chain = LinearChain(ions=[ion], comfrequencies=(x=3e6, y=3e6, z=trap_freq), selectedmodes=(; z=[1]))
    trap_pre = Chamber(iontrap=chain, B=1e-13, Bhat=ẑ)
    transition_λ = transitionwavelength(ion, ("g", "e"), trap_pre)
    eta = ((2 * π / transition_λ) / sqrt(2)) * sqrt(hbar / (2 * m_Ca40 * 2 * π * trap_freq))
    dummy_laser = Laser(I=laser_intensity, k=(x̂ + ẑ) / √2, ϵ=ŷ, λ=transition_λ)
    Omega_eff = IonSim.matrixelement(ion, ("g", "e"), laser_intensity, polarization(dummy_laser), wavevector(dummy_laser), (; x=0.0, y=0.0, z=1.0)) * 2 * π
    
    t_sig_s = (2 * beta_signal) / (eta * Omega_eff)
    Gamma_t = gamma_phi * t_sig_s

    sim_mode = zmodes(trap_pre)[1]
    sim_mode.N = 50; sim_mode.shape = [51]
    initial_sys_origin = normalize(ion["g"]) ⊗ Ket(sim_mode, fockstate(FockBasis(50), n_fock).data)

    println("\nEvaluating ODE Simulation...")
    noisy_beta_state = SDF_apply_with_alpha_dephasing(laser_intensity, 0.0, beta_signal, false, initial_sys_origin, n_fock, gamma_phi, beta_signal)
    noisy_beta_state_pert = SDF_apply_with_alpha_dephasing(laser_intensity, 0.0, beta_signal + d_beta, false, initial_sys_origin, n_fock, gamma_phi, beta_signal)

    println("Evaluating Stepwise Trotter Theory & Static β_eff Theory...")
    b_theory = FockBasis(50)
    a_op = QuantumOpticsBase.destroy(b_theory)
    
    # Generate Trotter States and shift to Readout Frame
    rho_trotter = get_stepwise_combined_theory_rho(n_fock, Gamma_t, cool, heat, beta_signal, steps)
    rho_trotter_pert = get_stepwise_combined_theory_rho(n_fock, Gamma_t, cool, heat, beta_signal + d_beta, steps)
    D_shift_trotter = QuantumOpticsBase.displace(b_theory, -expect(a_op, rho_trotter))
    rho_trotter_shift = D_shift_trotter * rho_trotter * dagger(D_shift_trotter)
    rho_trotter_pert_shift = D_shift_trotter * rho_trotter_pert * dagger(D_shift_trotter)

    # Generate Static β_eff States and shift to Readout Frame
    rho_static = get_static_open_theory_rho(n_fock, Gamma_t, beta_signal)
    rho_static_pert = get_static_open_theory_rho(n_fock, Gamma_t, beta_signal + d_beta)
    D_shift_static = QuantumOpticsBase.displace(b_theory, -expect(a_op, rho_static))
    rho_static_shift = D_shift_static * rho_static * dagger(D_shift_static)
    rho_static_pert_shift = D_shift_static * rho_static_pert * dagger(D_shift_static)

    # Tensor all theoretical states exactly as before
    sys_trotter = dm(normalize(ion["g"])) ⊗ DenseOperator(sim_mode, sim_mode, rho_trotter_shift.data)
    sys_trotter_pert = dm(normalize(ion["g"])) ⊗ DenseOperator(sim_mode, sim_mode, rho_trotter_pert_shift.data)
    sys_static = dm(normalize(ion["g"])) ⊗ DenseOperator(sim_mode, sim_mode, rho_static_shift.data)
    sys_static_pert = dm(normalize(ion["g"])) ⊗ DenseOperator(sim_mode, sim_mode, rho_static_pert_shift.data)

    alphas = collect(0.1:0.01:1.6)
    sim_qcrbs, trotter_qcrbs, static_qcrbs, closed_qcrbs = Float64[], Float64[], Float64[], Float64[]

    println("Sweeping α (Evaluating using rigorous readout pulse)...")
    for a in alphas
        # 1. ODE Simulation
        state_a_sim = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, noisy_beta_state, n_fock, gamma_phi, beta_signal)
        state_a_pert_sim = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, noisy_beta_state_pert, n_fock, gamma_phi, beta_signal)
        push!(sim_qcrbs, calculate_qcrb(ptrace(state_a_sim, 2), ptrace(state_a_pert_sim, 2), d_beta))

        # 2. Trotter Expansion (Exact Readout)
        state_a_trotter = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, sys_trotter, n_fock, gamma_phi, beta_signal)
        state_a_pert_trotter = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, sys_trotter_pert, n_fock, gamma_phi, beta_signal)
        push!(trotter_qcrbs, calculate_qcrb(ptrace(state_a_trotter, 2), ptrace(state_a_pert_trotter, 2), d_beta))
        
        # 3. Static Beta_eff Approx (Exact Readout)
        state_a_static = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, sys_static, n_fock, gamma_phi, beta_signal)
        state_a_pert_static = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, sys_static_pert, n_fock, gamma_phi, beta_signal)
        push!(static_qcrbs, calculate_qcrb(ptrace(state_a_static, 2), ptrace(state_a_pert_static, 2), d_beta))

        # 4. Ideal Closed System
        L_val = laguerre(n_fock, 4 * a^2)
        push!(closed_qcrbs, exp(4 * a^2) / (16 * a^2 * L_val^2))
    end

    fig = figure(figsize=(10, 6.5))
    
    plot(alphas, sim_qcrbs, "o", markersize=8, color="dodgerblue", linewidth=0, label="Simulation (master_dynamic ODE)")
    plot(alphas, trotter_qcrbs, "-", color="forestgreen", linewidth=2.5, alpha=0.85, label="Theory (Stepwise Trotter Expansion)")
    plot(alphas, static_qcrbs, "--", color="darkorange", linewidth=2.5, alpha=0.9, label=L"Theory (Static $\beta_{eff}$ Approx)")
    plot(alphas, closed_qcrbs, "r:", linewidth=2.5, alpha=0.8, label="Theory (Ideal Closed System)")

    yscale("log")
    ylim(1e-1, 2e3)
    xlabel("Readout Displacement [α]", fontsize=12)
    ylabel("QCRB (Ancilla Measurement Limit)", fontsize=12)
    title("Quad-Plot Analysis: Macroscopic β=$(beta_signal), Fock = $(n_fock), Γ_ϕ = $(gamma_phi) Hz", fontsize=13)
    
    legend(fontsize=10, loc="upper left")
    grid(alpha=0.4)
    
    display(fig)
end

plot_qcrb_macroscopic_beta_step(1, 1e3, 365.0, 0.1, 0.0, 2.0, 10)

InitError: InitError: PyError (PyImport_ImportModule

The Python package matplotlib could not be imported by pyimport. Usually this means
that you did not install matplotlib in the Python version being used by PyCall.

PyCall is currently configured to use the Python version at:

/usr/bin/python3

and you should use whatever mechanism you usually use (apt-get, pip, conda,
etcetera) to install the Python package containing the matplotlib module.

One alternative is to re-configure PyCall to use a different Python
version on your system: set ENV["PYTHON"] to the path/name of the python
executable you want to use, run Pkg.build("PyCall"), and re-launch Julia.

Another alternative is to configure PyCall to use a Julia-specific Python
distribution via the Conda.jl package (which installs a private Anaconda
Python distribution), which has the advantage that packages can be installed
and kept up-to-date via Julia.  As explained in the PyCall documentation,
set ENV["PYTHON"]="", run Pkg.build(

In [2]:
ENV["MPLBACKEND"] = "agg"
using QuantumOptics
using PyPlot
using LinearAlgebra
using IonSim

# -------------------------------------------------------------
# Math Helpers & Exact QCRB Eigen-Solver
# -------------------------------------------------------------
function laguerre(n::Int, x::Float64)
    if n == 0 return 1.0 end
    if n == 1 return 1.0 - x end
    L_prev2 = 1.0
    L_prev1 = 1.0 - x
    L_curr = 0.0
    for k in 1:(n-1)
        L_curr = ((2k + 1 - x) * L_prev1 - k * L_prev2) / (k + 1)
        L_prev2 = L_prev1
        L_prev1 = L_curr
    end
    return L_curr
end

function get_exact_beta_eff(beta_target::Float64, Gamma_t::Float64)
    if Gamma_t < 1e-5 return beta_target / sqrt(3.0) end
    x = Gamma_t / 2.0
    term1 = 1.0
    term2 = (2.0 / x) * (exp(-x) - 1.0)
    term3 = -(1.0 / (2.0 * x)) * (exp(-2.0 * x) - 1.0)
    avg_sq = (beta_target^2 / x^2) * (term1 + term2 + term3)
    return sqrt(max(0.0, avg_sq))
end

function calculate_qcrb(state, state_perturbed, d_alpha::Float64; tol=1e-10)
    rho_dense_data = dense(state).data
    rho_pert_dense_data = dense(state_perturbed).data
    drho_data = (rho_pert_dense_data - rho_dense_data) / d_alpha
    λ, V = LinearAlgebra.eigen(Hermitian(rho_dense_data))
    
    qfi = 0.0
    n = length(λ)
    for i in 1:n
        for j in 1:n
            if real(λ[i] + λ[j]) > tol
                @views v_i = V[:, i]
                @views v_j = V[:, j]
                drho_ij = dot(v_i, drho_data * v_j)
                qfi += 2 * (abs(drho_ij)^2) / real(λ[i] + λ[j])
            end
        end
    end
    return qfi > 0 ? 1.0 / qfi : Inf
end

# -------------------------------------------------------------
# 1. Simulation ODE Logic (Forced Keyword Args for Thermal Channels)
# -------------------------------------------------------------
function SDF_apply_with_alpha_dephasing(laser_intensity = 1e4, angle = 0.0, target_disp = 0.716, is_SDF = false, initial_state = nothing, fock_state_n = 0, gamma_phi_set = 1e-2, baseline_beta = 1.0; gamma_cool_set = 0.0, gamma_heat_set = 0.0)
    trap_freq = 19e5 
    ion = Ca40([("S1/2", -1/2, "g"), ("D5/2", -5/2, "e")])
    chain = LinearChain(ions=[ion], comfrequencies=(x=3e6, y=3e6, z=trap_freq), selectedmodes=(; z=[1]))
    mode = zmodes(chain)[1] 
    mode.N = 50; mode.shape = [51]
    
    k_vec = (x̂ + ẑ) / √2
    ϵ_vec = ŷ
    B_field = 1e-13
    trap_pre = Chamber(iontrap=chain, B=B_field, Bhat=ẑ)
    transition_λ = transitionwavelength(ion, ("g", "e"), trap_pre)
    
    ψ_initial = initial_state !== nothing ? initial_state : (normalize(ion["g"]) ⊗ fockstate(mode, fock_state_n))

    hbar = 1.0545718e-34; amu = 1.66053906e-27; m_Ca40 = 39.96259 * amu    
    eta = ((2 * π / transition_λ) / sqrt(2)) * sqrt(hbar / (2 * m_Ca40 * 2 * π * trap_freq))
    dummy_laser = Laser(I=laser_intensity, k=k_vec, ϵ=ϵ_vec, λ=transition_λ)
    Omega_eff = IonSim.matrixelement(ion, ("g", "e"), laser_intensity, polarization(dummy_laser), wavevector(dummy_laser), (; x=0.0, y=0.0, z=1.0)) * 2 * π

    if is_SDF
        alpha_complex = target_disp * exp(1im * angle)
        g, e = ion["g"], ion["e"]
        x_plus, x_minus = normalize(g + e), normalize(g - e)
        
        P_plus, P_minus = x_plus ⊗ dagger(x_plus), x_minus ⊗ dagger(x_minus)
        D_plus, D_minus = QuantumOptics.displace(mode, alpha_complex), QuantumOptics.displace(mode, -alpha_complex)
        
        U_SDF = (P_plus ⊗ D_plus) + (P_minus ⊗ D_minus)
        return isa(ψ_initial, Operator) ? U_SDF * ψ_initial * dagger(U_SDF) : U_SDF * ψ_initial
    else
        I_spin = identityoperator(ion["g"].basis)  
        
        gamma_phi_us = gamma_phi_set * 1e-6 
        gamma_cool_us = gamma_cool_set * 1e-6
        gamma_heat_us = gamma_heat_set * 1e-6

        b_fock = FockBasis(mode.N) 
        a_op = DenseOperator(mode, mode, QuantumOptics.destroy(b_fock).data)
        at_op = DenseOperator(mode, mode, QuantumOptics.create(b_fock).data)
        n_op = DenseOperator(mode, mode, QuantumOptics.number(b_fock).data)

        target_time_s = (2 * baseline_beta) / (eta * Omega_eff)
        target_time_us = target_time_s * 1e6

        drive_rate = target_disp / target_time_us
        H_m = (1im * drive_rate) * (exp(1im * angle) * at_op - exp(-1im * angle) * a_op)
        H_sys = I_spin ⊗ H_m

        J_ops = typeof(H_sys)[] # Prevent typemismatches in QuantumOptics
        
        if gamma_phi_set > 0
            push!(J_ops, sqrt(gamma_phi_us) * (I_spin ⊗ n_op))
        end
        if gamma_cool_set > 0
            push!(J_ops, sqrt(gamma_cool_us) * (I_spin ⊗ a_op))
        end
        if gamma_heat_set > 0
            push!(J_ops, sqrt(gamma_heat_us) * (I_spin ⊗ at_op))
        end

        f_dyn(time_t, rho_state) = (H_sys, J_ops, dagger.(J_ops))

        rho_initial = isa(ψ_initial, Ket) ? dm(ψ_initial) : ψ_initial
        _, sol_beta = timeevolution.master_dynamic([0.0, target_time_us], rho_initial, f_dyn, reltol=1e-9, abstol=1e-11)
        return sol_beta[end]
    end
end

# -------------------------------------------------------------
# 2. Theory: Unified Stepwise Open System
# -------------------------------------------------------------
function get_stepwise_combined_theory_rho(n_fock::Int, Gamma_phi_t::Float64, Gamma_cool_t::Float64, Gamma_heat_t::Float64, beta_target::Float64, steps::Int=500)
    N_cut = 50 # Must be exactly 50 to match sim_mode.N
    b = FockBasis(N_cut)
    rho = dm(fockstate(b, n_fock))
    
    a_op = QuantumOptics.destroy(b)
    at_op = QuantumOptics.create(b)
    n_op = QuantumOptics.number(b)
    I_op = QuantumOptics.identityoperator(b)
    
    d_beta = beta_target / steps
    d_g_phi = Gamma_phi_t / steps
    d_g_c = Gamma_cool_t / steps
    d_g_h = Gamma_heat_t / steps
    
    D_step = QuantumOptics.displace(b, d_beta)
    D_step_dag = QuantumOptics.dagger(D_step)
    
    M0_c = I_op - (d_g_c / 2.0) * n_op
    M1_c = sqrt(d_g_c) * a_op
    
    M0_h = I_op - (d_g_h / 2.0) * (n_op + I_op)
    M1_h = sqrt(d_g_h) * at_op
    
    for k in 1:steps
        rho = D_step * rho * D_step_dag
        
        if d_g_phi > 0
            for i in 1:(N_cut + 1)
                for j in 1:(N_cut + 1)
                    rho.data[i, j] *= exp(-0.5 * d_g_phi * ((i - 1) - (j - 1))^2)
                end
            end
        end
        if d_g_c > 0
            rho = M0_c * rho * dagger(M0_c) + M1_c * rho * dagger(M1_c)
        end
        if d_g_h > 0
            rho = M0_h * rho * dagger(M0_h) + M1_h * rho * dagger(M1_h)
        end
    end
    return rho
end

# -------------------------------------------------------------
# 3. Theory: Static β_eff Approximation
# -------------------------------------------------------------
function get_static_open_theory_rho(n_fock::Int, Gamma_t::Float64, beta_target::Float64)
    N_cut = 50 
    b = FockBasis(N_cut)
    
    beta_eff = get_exact_beta_eff(beta_target, Gamma_t)
    D_eff = QuantumOptics.displace(b, beta_eff)
    rho = D_eff * dm(fockstate(b, n_fock)) * dagger(D_eff)
    
    for i in 1:(N_cut + 1)
        for j in 1:(N_cut + 1)
            rho.data[i, j] *= exp(-0.5 * Gamma_t * ((i - 1) - (j - 1))^2)
        end
    end
    return rho
end

# -------------------------------------------------------------
# Quad-Plot Main Evaluation Function
# -------------------------------------------------------------
function plot_qcrb_macroscopic_beta_step(n_fock::Int=1, laser_intensity::Float64=1e3, gamma_phi::Float64=365.0, cool::Float64=0.0, heat::Float64=0.0, beta_signal::Float64=2.0, steps::Int=50)
    d_beta = 1e-4 
    
    trap_freq = 19e5; hbar = 1.0545718e-34; amu = 1.66053906e-27; m_Ca40 = 39.96259 * amu    
    ion = Ca40([("S1/2", -1/2, "g"), ("D5/2", -5/2, "e")])
    chain = LinearChain(ions=[ion], comfrequencies=(x=3e6, y=3e6, z=trap_freq), selectedmodes=(; z=[1]))
    trap_pre = Chamber(iontrap=chain, B=1e-13, Bhat=ẑ)
    transition_λ = transitionwavelength(ion, ("g", "e"), trap_pre)
    eta = ((2 * π / transition_λ) / sqrt(2)) * sqrt(hbar / (2 * m_Ca40 * 2 * π * trap_freq))
    dummy_laser = Laser(I=laser_intensity, k=(x̂ + ẑ) / √2, ϵ=ŷ, λ=transition_λ)
    Omega_eff = IonSim.matrixelement(ion, ("g", "e"), laser_intensity, polarization(dummy_laser), wavevector(dummy_laser), (; x=0.0, y=0.0, z=1.0)) * 2 * π
    
    t_sig_s = (2 * beta_signal) / (eta * Omega_eff)
    Gamma_phi_t = gamma_phi * t_sig_s
    Gamma_cool_t = cool * t_sig_s
    Gamma_heat_t = heat * t_sig_s

    sim_mode = zmodes(trap_pre)[1]
    sim_mode.N = 50; sim_mode.shape = [51]
    initial_sys_origin = normalize(ion["g"]) ⊗ Ket(sim_mode, fockstate(FockBasis(50), n_fock).data)

    println("\nEvaluating ODE Simulation...")
    # Feed thermal rates through keyword arguments to prevent positional shifting
    noisy_beta_state = SDF_apply_with_alpha_dephasing(laser_intensity, 0.0, beta_signal, false, initial_sys_origin, n_fock, gamma_phi, beta_signal; gamma_cool_set=cool, gamma_heat_set=heat)
    noisy_beta_state_pert = SDF_apply_with_alpha_dephasing(laser_intensity, 0.0, beta_signal + d_beta, false, initial_sys_origin, n_fock, gamma_phi, beta_signal; gamma_cool_set=cool, gamma_heat_set=heat)

    println("Evaluating Stepwise Trotter Theory & Static β_eff Theory...")
    b_theory = FockBasis(50)
    a_op = QuantumOpticsBase.destroy(b_theory)
    
    rho_trotter = get_stepwise_combined_theory_rho(n_fock, Gamma_phi_t, Gamma_cool_t, Gamma_heat_t, beta_signal, steps)
    rho_trotter_pert = get_stepwise_combined_theory_rho(n_fock, Gamma_phi_t, Gamma_cool_t, Gamma_heat_t, beta_signal + d_beta, steps)
    D_shift_trotter = QuantumOpticsBase.displace(b_theory, -expect(a_op, rho_trotter))
    rho_trotter_shift = D_shift_trotter * rho_trotter * dagger(D_shift_trotter)
    rho_trotter_pert_shift = D_shift_trotter * rho_trotter_pert * dagger(D_shift_trotter)

    rho_static = get_static_open_theory_rho(n_fock, Gamma_phi_t, beta_signal)
    rho_static_pert = get_static_open_theory_rho(n_fock, Gamma_phi_t, beta_signal + d_beta)
    D_shift_static = QuantumOpticsBase.displace(b_theory, -expect(a_op, rho_static))
    rho_static_shift = D_shift_static * rho_static * dagger(D_shift_static)
    rho_static_pert_shift = D_shift_static * rho_static_pert * dagger(D_shift_static)

    sys_trotter = dm(normalize(ion["g"])) ⊗ DenseOperator(sim_mode, sim_mode, rho_trotter_shift.data)
    sys_trotter_pert = dm(normalize(ion["g"])) ⊗ DenseOperator(sim_mode, sim_mode, rho_trotter_pert_shift.data)
    sys_static = dm(normalize(ion["g"])) ⊗ DenseOperator(sim_mode, sim_mode, rho_static_shift.data)
    sys_static_pert = dm(normalize(ion["g"])) ⊗ DenseOperator(sim_mode, sim_mode, rho_static_pert_shift.data)

    alphas = collect(0.1:0.02:1.6)
    sim_qcrbs, trotter_qcrbs, static_qcrbs, closed_qcrbs = Float64[], Float64[], Float64[], Float64[]

    println("Sweeping α (Evaluating using rigorous readout pulse)...")
    for a in alphas
        # Note: No thermal noise during the instantaneous readout pulse (using positional args limits to pure dephasing defaults)
        state_a_sim = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, noisy_beta_state, n_fock, gamma_phi, beta_signal)
        state_a_pert_sim = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, noisy_beta_state_pert, n_fock, gamma_phi, beta_signal)
        push!(sim_qcrbs, calculate_qcrb(ptrace(state_a_sim, 2), ptrace(state_a_pert_sim, 2), d_beta))

        state_a_trotter = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, sys_trotter, n_fock, gamma_phi, beta_signal)
        state_a_pert_trotter = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, sys_trotter_pert, n_fock, gamma_phi, beta_signal)
        push!(trotter_qcrbs, calculate_qcrb(ptrace(state_a_trotter, 2), ptrace(state_a_pert_trotter, 2), d_beta))
        
        state_a_static = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, sys_static, n_fock, gamma_phi, beta_signal)
        state_a_pert_static = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, sys_static_pert, n_fock, gamma_phi, beta_signal)
        push!(static_qcrbs, calculate_qcrb(ptrace(state_a_static, 2), ptrace(state_a_pert_static, 2), d_beta))

        L_val = laguerre(n_fock, 4 * a^2)
        push!(closed_qcrbs, exp(4 * a^2) / (16 * a^2 * L_val^2))
    end

    fig = figure(figsize=(10, 6.5))
    
    plot(alphas, sim_qcrbs, "o", markersize=8, color="dodgerblue", linewidth=0, label="Simulation (master_dynamic ODE)")
    plot(alphas, trotter_qcrbs, "-", color="forestgreen", linewidth=2.5, alpha=0.85, label="Theory (Stepwise Trotter Expansion)")
    plot(alphas, static_qcrbs, "--", color="darkorange", linewidth=2.5, alpha=0.9, label=L"Theory (Static $\beta_{eff}$ Approx)")
    plot(alphas, closed_qcrbs, "r:", linewidth=2.5, alpha=0.8, label="Theory (Ideal Closed System)")

    yscale("log")
    ylim(1e-1, 2e3)
    xlabel("Readout Displacement [α]", fontsize=12)
    ylabel("QCRB (Ancilla Measurement Limit)", fontsize=12)
    title("Quad-Plot: Macroscopic β=$(beta_signal), N=$(steps), Γ_ϕ=$(gamma_phi), Γ_c=$(cool), Γ_h=$(heat) Hz", fontsize=13)
    
    legend(fontsize=10, loc="upper left")
    grid(alpha=0.4)
    
    display(fig)
end

plot_qcrb_macroscopic_beta_step(1, 1e3, 365.0, 50.0, 10.0, 2.0, 30)

ArgumentError: ArgumentError: Package IonSim not found in current path.
- Run `import Pkg; Pkg.add("IonSim")` to install the IonSim package.

In [3]:
plot_qcrb_macroscopic_beta_step(1, 1e3, 0.0, 50.0, 10.0, 2.0, 30)

UndefVarError: UndefVarError: `plot_qcrb_macroscopic_beta_step` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [4]:
plot_qcrb_macroscopic_beta_step(1, 1e3, 0.0, 50.0, 0.0, 2.0, 30)

UndefVarError: UndefVarError: `plot_qcrb_macroscopic_beta_step` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [5]:
plot_qcrb_macroscopic_beta_step(1, 1e3, 0.0, 0.0, 0.0, 2.0, 30)

UndefVarError: UndefVarError: `plot_qcrb_macroscopic_beta_step` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [6]:
plot_qcrb_macroscopic_beta_step(1, 1e3, 365.0, 0.0, 0.0, 2.0, 2)

UndefVarError: UndefVarError: `plot_qcrb_macroscopic_beta_step` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [7]:
plot_qcrb_macroscopic_beta_step(1, 1e3, 365.0, 0.0, 0.0, 2.0, 1)

UndefVarError: UndefVarError: `plot_qcrb_macroscopic_beta_step` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [8]:
plot_qcrb_macroscopic_beta_step(1, 1e3, 365.0, 0.0, 0.0, 2.0, 10)

UndefVarError: UndefVarError: `plot_qcrb_macroscopic_beta_step` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [9]:
function analyze_trotter_convergence(n_fock::Int=1, laser_intensity::Float64=1e3, gamma_phi::Float64=365.0, cool::Float64=0.0, heat::Float64=0.0, beta_signal::Float64=2.0, N_values::AbstractVector{Int}=1:1:20)
    d_beta = 1e-4 
    alphas = collect(0.1:0.02:1.6)
    
    # -------------------------------------------------------------
    # 1. Physical System Setup
    # -------------------------------------------------------------
    trap_freq = 19e5; hbar = 1.0545718e-34; amu = 1.66053906e-27; m_Ca40 = 39.96259 * amu    
    ion = Ca40([("S1/2", -1/2, "g"), ("D5/2", -5/2, "e")])
    chain = LinearChain(ions=[ion], comfrequencies=(x=3e6, y=3e6, z=trap_freq), selectedmodes=(; z=[1]))
    trap_pre = Chamber(iontrap=chain, B=1e-13, Bhat=ẑ)
    transition_λ = transitionwavelength(ion, ("g", "e"), trap_pre)
    eta = ((2 * π / transition_λ) / sqrt(2)) * sqrt(hbar / (2 * m_Ca40 * 2 * π * trap_freq))
    
    dummy_laser = Laser(I=laser_intensity, k=(x̂ + ẑ) / √2, ϵ=ŷ, λ=transition_λ)
    Omega_eff = IonSim.matrixelement(ion, ("g", "e"), laser_intensity, polarization(dummy_laser), wavevector(dummy_laser), (; x=0.0, y=0.0, z=1.0)) * 2 * π
    
    t_sig_s = (2 * beta_signal) / (eta * Omega_eff)
    Gamma_phi_t = gamma_phi * t_sig_s
    Gamma_cool_t = cool * t_sig_s
    Gamma_heat_t = heat * t_sig_s

    sim_mode = zmodes(trap_pre)[1]
    sim_mode.N = 50; sim_mode.shape = [51]
    initial_sys_origin = normalize(ion["g"]) ⊗ Ket(sim_mode, fockstate(FockBasis(50), n_fock).data)
    b_theory = FockBasis(50)
    a_op = QuantumOpticsBase.destroy(b_theory)

    # -------------------------------------------------------------
    # 2. ODE Simulation (Ground Truth Baseline - Runs Once)
    # -------------------------------------------------------------
    println("Evaluating baseline ODE Simulation...")
    noisy_beta_state = SDF_apply_with_alpha_dephasing(laser_intensity, 0.0, beta_signal, false, initial_sys_origin, n_fock, gamma_phi, beta_signal; gamma_cool_set=cool, gamma_heat_set=heat)
    noisy_beta_state_pert = SDF_apply_with_alpha_dephasing(laser_intensity, 0.0, beta_signal + d_beta, false, initial_sys_origin, n_fock, gamma_phi, beta_signal; gamma_cool_set=cool, gamma_heat_set=heat)

    sim_qcrbs = Float64[]
    for a in alphas
        state_a_sim = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, noisy_beta_state, n_fock, gamma_phi, beta_signal)
        state_a_pert_sim = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, noisy_beta_state_pert, n_fock, gamma_phi, beta_signal)
        push!(sim_qcrbs, calculate_qcrb(ptrace(state_a_sim, 2), ptrace(state_a_pert_sim, 2), d_beta))
    end
    
    min_qcrb_sim = minimum(sim_qcrbs)
    opt_alpha_sim = alphas[argmin(sim_qcrbs)]

    # -------------------------------------------------------------
    # 3. Trotter Expansion Sweep (Loops over N_values)
    # -------------------------------------------------------------
    qcrb_diffs = Float64[]
    opt_alphas_trotter = Float64[]

    println("Sweeping Trotter steps N...")
    for N in N_values
        rho_trotter = get_stepwise_combined_theory_rho(n_fock, Gamma_phi_t, Gamma_cool_t, Gamma_heat_t, beta_signal, N)
        rho_trotter_pert = get_stepwise_combined_theory_rho(n_fock, Gamma_phi_t, Gamma_cool_t, Gamma_heat_t, beta_signal + d_beta, N)
        
        D_shift_trotter = QuantumOpticsBase.displace(b_theory, -expect(a_op, rho_trotter))
        rho_trotter_shift = D_shift_trotter * rho_trotter * dagger(D_shift_trotter)
        rho_trotter_pert_shift = D_shift_trotter * rho_trotter_pert * dagger(D_shift_trotter)

        sys_trotter = dm(normalize(ion["g"])) ⊗ DenseOperator(sim_mode, sim_mode, rho_trotter_shift.data)
        sys_trotter_pert = dm(normalize(ion["g"])) ⊗ DenseOperator(sim_mode, sim_mode, rho_trotter_pert_shift.data)

        trotter_qcrbs = Float64[]
        for a in alphas
            state_a_trotter = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, sys_trotter, n_fock, gamma_phi, beta_signal)
            state_a_pert_trotter = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, sys_trotter_pert, n_fock, gamma_phi, beta_signal)
            push!(trotter_qcrbs, calculate_qcrb(ptrace(state_a_trotter, 2), ptrace(state_a_pert_trotter, 2), d_beta))
        end
        
        min_qcrb_trotter = minimum(trotter_qcrbs)
        push!(opt_alphas_trotter, alphas[argmin(trotter_qcrbs)])
        push!(qcrb_diffs, abs(min_qcrb_trotter - min_qcrb_sim))
    end

    # -------------------------------------------------------------
    # 4. Plotting the Convergence
    # -------------------------------------------------------------
    fig, (ax1, ax2) = subplots(2, 1, figsize=(10, 8), sharex=true)
    
    # Subplot 1: Minimum QCRB Difference
    ax1.plot(N_values, qcrb_diffs, "o-", color="purple", linewidth=2.5, markersize=7, label=L"|$\min(\text{QCRB}_{Trotter}) - \min(\text{QCRB}_{ODE})$|")
    ax1.set_yscale("log")
    ax1.set_ylabel("Δ min(QCRB)", fontsize=12)
    ax1.set_title("Trotter Convergence vs Master Equation ODE\n(Macroscopic β=$(beta_signal), Γ_ϕ=$(gamma_phi), Γ_c=$(cool), Γ_h=$(heat) Hz)", fontsize=13)
    ax1.legend(loc="upper right")
    ax1.grid(alpha=0.4)

    # Subplot 2: Optimal Alpha Comparison
    ax2.plot(N_values, opt_alphas_trotter, "s-", color="forestgreen", linewidth=2.5, markersize=7, label=L"Trotter Optimal $\alpha_{opt}$")
    ax2.axhline(y=opt_alpha_sim, color="dodgerblue", linestyle="--", linewidth=2.5, label=L"ODE Simulation Baseline $\alpha_{opt}$")
    ax2.set_xlabel("Number of Trotter Steps (N)", fontsize=12)
    ax2.set_ylabel(L"Optimal Displacement [$\alpha_{opt}$]", fontsize=12)
    ax2.legend(loc="upper right")
    ax2.grid(alpha=0.4)
    
    tight_layout()
    display(fig)
end

# Example usage:
# analyze_trotter_convergence(1, 1e3, 365.0, 0.0, 0.0, 2.0, 1:1:15)

analyze_trotter_convergence (generic function with 8 methods)

In [10]:
analyze_trotter_convergence(1, 1e3, 365.0, 0.0, 0.0, 2.0, 1:1:15)

UndefVarError: UndefVarError: `Ca40` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [11]:
analyze_trotter_convergence(1, 1e3, 365.0, 0.0, 0.0, 2.0, 1:1:30)

UndefVarError: UndefVarError: `Ca40` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [12]:
analyze_trotter_convergence(1, 1e3, 0.0, 50.0, 0.0, 2.0, 1:1:30)

UndefVarError: UndefVarError: `Ca40` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [13]:
plot_qcrb_macroscopic_beta_step(1, 1e3, 0.0, 50.0, 0.0, 2.0, 1)

UndefVarError: UndefVarError: `plot_qcrb_macroscopic_beta_step` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [14]:
plot_qcrb_macroscopic_beta_step(1, 1e3, 0.0, 50.0, 0.0, 2.0, 2)

UndefVarError: UndefVarError: `plot_qcrb_macroscopic_beta_step` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [15]:
plot_qcrb_macroscopic_beta_step(1, 1e3, 365.0, 50.0, 0.0, 2.0, 1)

UndefVarError: UndefVarError: `plot_qcrb_macroscopic_beta_step` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [16]:
plot_qcrb_macroscopic_beta_step(1, 1e3, 365.0, 50.0, 0.0, 2.0, 10)

UndefVarError: UndefVarError: `plot_qcrb_macroscopic_beta_step` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [17]:
using Printf

function analyze_trotter_convergence(n_fock::Int=1, laser_intensity::Float64=1e3, gamma_phi::Float64=365.0, cool::Float64=0.0, heat::Float64=0.0, beta_signal::Float64=2.0, N_values::AbstractVector{Int}=1:1:20)
    d_beta = 1e-4 
    alphas = collect(0.1:0.02:1.6)
    
    # -------------------------------------------------------------
    # 1. Physical System Setup & Dimensionless Noise Calculation
    # -------------------------------------------------------------
    trap_freq = 19e5; hbar = 1.0545718e-34; amu = 1.66053906e-27; m_Ca40 = 39.96259 * amu    
    ion = Ca40([("S1/2", -1/2, "g"), ("D5/2", -5/2, "e")])
    chain = LinearChain(ions=[ion], comfrequencies=(x=3e6, y=3e6, z=trap_freq), selectedmodes=(; z=[1]))
    trap_pre = Chamber(iontrap=chain, B=1e-13, Bhat=ẑ)
    transition_λ = transitionwavelength(ion, ("g", "e"), trap_pre)
    eta = ((2 * π / transition_λ) / sqrt(2)) * sqrt(hbar / (2 * m_Ca40 * 2 * π * trap_freq))
    
    dummy_laser = Laser(I=laser_intensity, k=(x̂ + ẑ) / √2, ϵ=ŷ, λ=transition_λ)
    Omega_eff = IonSim.matrixelement(ion, ("g", "e"), laser_intensity, polarization(dummy_laser), wavevector(dummy_laser), (; x=0.0, y=0.0, z=1.0)) * 2 * π
    
    # Calculate integrated dimensionless noise (Γ * t_sig)
    t_sig_s = (2 * beta_signal) / (eta * Omega_eff)
    Gamma_phi_t = gamma_phi * t_sig_s
    Gamma_cool_t = cool * t_sig_s
    Gamma_heat_t = heat * t_sig_s

    sim_mode = zmodes(trap_pre)[1]
    sim_mode.N = 50; sim_mode.shape = [51]
    initial_sys_origin = normalize(ion["g"]) ⊗ Ket(sim_mode, fockstate(FockBasis(50), n_fock).data)
    b_theory = FockBasis(50)
    a_op = QuantumOpticsBase.destroy(b_theory)

    # -------------------------------------------------------------
    # 2. ODE Simulation (Ground Truth Baseline - Runs Once)
    # -------------------------------------------------------------
    println("Evaluating baseline ODE Simulation...")
    noisy_beta_state = SDF_apply_with_alpha_dephasing(laser_intensity, 0.0, beta_signal, false, initial_sys_origin, n_fock, gamma_phi, beta_signal; gamma_cool_set=cool, gamma_heat_set=heat)
    noisy_beta_state_pert = SDF_apply_with_alpha_dephasing(laser_intensity, 0.0, beta_signal + d_beta, false, initial_sys_origin, n_fock, gamma_phi, beta_signal; gamma_cool_set=cool, gamma_heat_set=heat)

    sim_qcrbs = Float64[]
    for a in alphas
        state_a_sim = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, noisy_beta_state, n_fock, gamma_phi, beta_signal)
        state_a_pert_sim = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, noisy_beta_state_pert, n_fock, gamma_phi, beta_signal)
        push!(sim_qcrbs, calculate_qcrb(ptrace(state_a_sim, 2), ptrace(state_a_pert_sim, 2), d_beta))
    end
    
    min_qcrb_sim = minimum(sim_qcrbs)
    opt_alpha_sim = alphas[argmin(sim_qcrbs)]

    # -------------------------------------------------------------
    # 3. Trotter Expansion Sweep (Loops over N_values)
    # -------------------------------------------------------------
    qcrb_diffs = Float64[]
    opt_alphas_trotter = Float64[]

    println("Sweeping Trotter steps N...")
    for N in N_values
        rho_trotter = get_stepwise_combined_theory_rho(n_fock, Gamma_phi_t, Gamma_cool_t, Gamma_heat_t, beta_signal, N)
        rho_trotter_pert = get_stepwise_combined_theory_rho(n_fock, Gamma_phi_t, Gamma_cool_t, Gamma_heat_t, beta_signal + d_beta, N)
        
        D_shift_trotter = QuantumOpticsBase.displace(b_theory, -expect(a_op, rho_trotter))
        rho_trotter_shift = D_shift_trotter * rho_trotter * dagger(D_shift_trotter)
        rho_trotter_pert_shift = D_shift_trotter * rho_trotter_pert * dagger(D_shift_trotter)

        sys_trotter = dm(normalize(ion["g"])) ⊗ DenseOperator(sim_mode, sim_mode, rho_trotter_shift.data)
        sys_trotter_pert = dm(normalize(ion["g"])) ⊗ DenseOperator(sim_mode, sim_mode, rho_trotter_pert_shift.data)

        trotter_qcrbs = Float64[]
        for a in alphas
            state_a_trotter = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, sys_trotter, n_fock, gamma_phi, beta_signal)
            state_a_pert_trotter = SDF_apply_with_alpha_dephasing(laser_intensity, pi/2, a, true, sys_trotter_pert, n_fock, gamma_phi, beta_signal)
            push!(trotter_qcrbs, calculate_qcrb(ptrace(state_a_trotter, 2), ptrace(state_a_pert_trotter, 2), d_beta))
        end
        
        min_qcrb_trotter = minimum(trotter_qcrbs)
        push!(opt_alphas_trotter, alphas[argmin(trotter_qcrbs)])
        push!(qcrb_diffs, abs(min_qcrb_trotter - min_qcrb_sim))
    end

    # -------------------------------------------------------------
    # 4. Plotting the Convergence
    # -------------------------------------------------------------
    fig, (ax1, ax2) = subplots(2, 1, figsize=(10, 8), sharex=true)
    
    # Subplot 1: Minimum QCRB Difference
    ax1.plot(N_values, qcrb_diffs, "o-", color="purple", linewidth=2.5, markersize=7, label=L"|$\min(\text{QCRB}_{Trotter}) - \min(\text{QCRB}_{ODE})$|")
    ax1.set_yscale("log")
    ax1.set_ylabel("Δ min(QCRB)", fontsize=12)
    
    # Updated Title with Dimensionless Noise
    title_str = @sprintf("Trotter Convergence vs ODE\n(Macroscopic β=%.1f, Dimensionless Noise: Γ_ϕ*t=%.4f, Γ_c*t=%.4f, Γ_h*t=%.4f)", 
                         beta_signal, Gamma_phi_t, Gamma_cool_t, Gamma_heat_t)
    ax1.set_title(title_str, fontsize=13)
    ax1.legend(loc="upper right")
    ax1.grid(alpha=0.4)

    # Subplot 2: Optimal Alpha Comparison
    ax2.plot(N_values, opt_alphas_trotter, "s-", color="forestgreen", linewidth=2.5, markersize=7, label=L"Trotter Optimal $\alpha_{opt}$")
    ax2.axhline(y=opt_alpha_sim, color="dodgerblue", linestyle="--", linewidth=2.5, label=L"ODE Simulation Baseline $\alpha_{opt}$")
    ax2.set_xlabel("Number of Trotter Steps (N)", fontsize=12)
    ax2.set_ylabel(L"Optimal Displacement [$\alpha_{opt}$]", fontsize=12)
    ax2.legend(loc="upper right")
    ax2.grid(alpha=0.4)
    
    tight_layout()
    display(fig)

    # -------------------------------------------------------------
    # 5. Return Dimensionless Parameters
    # -------------------------------------------------------------
    return (
        Gamma_phi_t = Gamma_phi_t,
        Gamma_cool_t = Gamma_cool_t,
        Gamma_heat_t = Gamma_heat_t,
        t_sig_s = t_sig_s
    )
end

analyze_trotter_convergence (generic function with 8 methods)

In [18]:
analyze_trotter_convergence(1, 1e3, 0.0, 50.0, 0.0, 2.0, 1:1:30)

UndefVarError: UndefVarError: `Ca40` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [19]:
analyze_trotter_convergence(1, 1e3, 0.0, 500.0, 0.0, 2.0, 1:1:30)

UndefVarError: UndefVarError: `Ca40` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [20]:
plot_qcrb_macroscopic_beta_step(1, 1e3, 0.0, 500.0, 0.0, 2.0, 30)

UndefVarError: UndefVarError: `plot_qcrb_macroscopic_beta_step` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [21]:
plot_qcrb_macroscopic_beta_step(1, 1e3, 0.0, 500.0, 0.0, 2.0, 70)

UndefVarError: UndefVarError: `plot_qcrb_macroscopic_beta_step` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [22]:
using Printf

function compare_simulation_times(n_fock::Int=1, laser_intensity::Float64=1e3, gamma_phi::Float64=365.0, cool::Float64=50.0, heat::Float64=10.0, beta_signal::Float64=2.0, N_steps::Int=30)
    
    # 1. Physical System Setup
    trap_freq = 19e5; hbar = 1.0545718e-34; amu = 1.66053906e-27; m_Ca40 = 39.96259 * amu    
    ion = Ca40([("S1/2", -1/2, "g"), ("D5/2", -5/2, "e")])
    chain = LinearChain(ions=[ion], comfrequencies=(x=3e6, y=3e6, z=trap_freq), selectedmodes=(; z=[1]))
    trap_pre = Chamber(iontrap=chain, B=1e-13, Bhat=ẑ)
    transition_λ = transitionwavelength(ion, ("g", "e"), trap_pre)
    eta = ((2 * π / transition_λ) / sqrt(2)) * sqrt(hbar / (2 * m_Ca40 * 2 * π * trap_freq))
    
    dummy_laser = Laser(I=laser_intensity, k=(x̂ + ẑ) / √2, ϵ=ŷ, λ=transition_λ)
    Omega_eff = IonSim.matrixelement(ion, ("g", "e"), laser_intensity, polarization(dummy_laser), wavevector(dummy_laser), (; x=0.0, y=0.0, z=1.0)) * 2 * π
    
    t_sig_s = (2 * beta_signal) / (eta * Omega_eff)
    Gamma_phi_t = gamma_phi * t_sig_s
    Gamma_cool_t = cool * t_sig_s
    Gamma_heat_t = heat * t_sig_s

    sim_mode = zmodes(trap_pre)[1]
    sim_mode.N = 50; sim_mode.shape = [51]
    initial_sys_origin = normalize(ion["g"]) ⊗ Ket(sim_mode, fockstate(FockBasis(50), n_fock).data)

    # 2. JIT Compiler Warm-up (Required for accurate benchmarking)
    println("Performing JIT warm-up...")
    _ = SDF_apply_with_alpha_dephasing(laser_intensity, 0.0, beta_signal, false, initial_sys_origin, n_fock, gamma_phi, beta_signal; gamma_cool_set=cool, gamma_heat_set=heat)
    _ = get_stepwise_combined_theory_rho(n_fock, Gamma_phi_t, Gamma_cool_t, Gamma_heat_t, beta_signal, N_steps)

    # 3. Benchmark ODE Simulation
    println("Timing master_dynamic ODE integration...")
    ode_time = @elapsed begin
        SDF_apply_with_alpha_dephasing(laser_intensity, 0.0, beta_signal, false, initial_sys_origin, n_fock, gamma_phi, beta_signal; gamma_cool_set=cool, gamma_heat_set=heat)
    end

    # 4. Benchmark Stepwise Trotter Expansion
    println("Timing Stepwise Trotter Expansion (N=$N_steps)...")
    trotter_time = @elapsed begin
        get_stepwise_combined_theory_rho(n_fock, Gamma_phi_t, Gamma_cool_t, Gamma_heat_t, beta_signal, N_steps)
    end

    # 5. Output Results
    speedup = ode_time / trotter_time
    @printf("\n--- Benchmark Results ---\n")
    @printf("ODE Simulation Time:       %.5f seconds\n", ode_time)
    @printf("Trotter Simulation Time:   %.5f seconds\n", trotter_time)
    @printf("Computational Speedup:     %.1fx faster\n", speedup)

    return (ode_time=ode_time, trotter_time=trotter_time, speedup=speedup)
end

# Execute the benchmark
compare_simulation_times(1, 1e3, 365.0, 50.0, 10.0, 2.0, 30)

UndefVarError: UndefVarError: `Ca40` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
ENV["MPLBACKEND"] = "agg"
using QuantumOptics
using PyPlot
using LinearAlgebra

# -------------------------------------------------------------
# QCRB Calculation 
# -------------------------------------------------------------
function calculate_qcrb(state, state_perturbed, d_alpha::Float64; tol=1e-10)
    rho_dense_data = dense(state).data
    rho_pert_dense_data = dense(state_perturbed).data
    drho_data = (rho_pert_dense_data - rho_dense_data) / d_alpha
    λ, V = LinearAlgebra.eigen(Hermitian(rho_dense_data))
    
    qfi = 0.0
    n = length(λ)
    for i in 1:n
        for j in 1:n
            if real(λ[i] + λ[j]) > tol
                @views v_i = V[:, i]
                @views v_j = V[:, j]
                drho_ij = dot(v_i, drho_data * v_j)
                qfi += 2 * (abs(drho_ij)^2) / real(λ[i] + λ[j])
            end
        end
    end
    return qfi > 0 ? 1.0 / qfi : Inf
end

# -------------------------------------------------------------
# Stepwise Trotter Expansion (Generalized for arbitrary initial state)
# -------------------------------------------------------------
function get_stepwise_combined_theory_rho(initial_rho::Operator, Gamma_phi_t::Float64, Gamma_cool_t::Float64, Gamma_heat_t::Float64, beta_target::Float64, steps::Int=40)
    b = initial_rho.basis_l
    N_cut = b.N
    
    a_op = QuantumOptics.destroy(b)
    at_op = QuantumOptics.create(b)
    n_op = QuantumOptics.number(b)
    I_op = QuantumOptics.identityoperator(b)
    
    d_beta = beta_target / steps
    d_g_phi = Gamma_phi_t / steps
    d_g_c = Gamma_cool_t / steps
    d_g_h = Gamma_heat_t / steps
    
    D_step = QuantumOptics.displace(b, d_beta)
    D_step_dag = QuantumOptics.dagger(D_step)
    
    M0_c = I_op - (d_g_c / 2.0) * n_op
    M1_c = sqrt(d_g_c) * a_op
    
    M0_h = I_op - (d_g_h / 2.0) * (n_op + I_op)
    M1_h = sqrt(d_g_h) * at_op
    
    rho = deepcopy(initial_rho)
    
    for k in 1:steps
        rho = D_step * rho * D_step_dag
        
        if d_g_phi > 0
            for i in 1:(N_cut + 1)
                for j in 1:(N_cut + 1)
                    rho.data[i, j] *= exp(-0.5 * d_g_phi * ((i - 1) - (j - 1))^2)
                end
            end
        end
        if d_g_c > 0
            rho = M0_c * rho * dagger(M0_c) + M1_c * rho * dagger(M1_c)
        end
        if d_g_h > 0
            rho = M0_h * rho * dagger(M0_h) + M1_h * rho * dagger(M1_h)
        end
    end
    return rho
end

# -------------------------------------------------------------
# Readout Pulse (Instantaneous Unitary SDF)
# -------------------------------------------------------------
function apply_readout_SDF(sys_rho::Operator, alpha::Float64)
    # sys_rho is assumed to be Spin ⊗ Motion
    b_spin = sys_rho.basis_l.bases[1]
    b_motion = sys_rho.basis_l.bases[2]
    
    g = basisstate(b_spin, 1)
    e = basisstate(b_spin, 2)
    x_plus = normalize(g + e)
    x_minus = normalize(g - e)
    
    P_plus = x_plus ⊗ dagger(x_plus)
    P_minus = x_minus ⊗ dagger(x_minus)
    
    # pi/2 phase shift for readout
    alpha_complex = alpha * exp(1im * pi/2)
    D_plus = QuantumOptics.displace(b_motion, alpha_complex)
    D_minus = QuantumOptics.displace(b_motion, -alpha_complex)
    
    U_SDF = (P_plus ⊗ D_plus) + (P_minus ⊗ D_minus)
    return U_SDF * sys_rho * dagger(U_SDF)
end

# -------------------------------------------------------------
# Phase Diagram Plotter (Sweeping n, r, and a target noise channel)
# -------------------------------------------------------------
function plot_multichannel_phase_diagram(; sweep_channel::Symbol=:dephasing, beta_signal::Float64=2.0)
    # Define the sweep ranges
    g_min_sweep = 1e-3
    g_max_sweep = 1e1
    sweep_axis = 10.0 .^ range(log10(g_min_sweep), log10(g_max_sweep), length=25)
    
    # Squeezing and Fock state configurations
    n_vals = [0, 1, 2, 3]
    r_vals = [0.0, 0.2, 0.4]
    
    # Fixed base noise (Dimensionless Γ*t)
    base_dephasing = 0.0
    base_cooling = 0.1
    base_heating = 0.5
    
    # Alphas to optimize over for QCRB
    alphas = collect(0.1:0.02:2.0)
    d_beta = 1e-4
    
    # Bases
    b_spin = SpinBasis(1//2)
    b_motion = FockBasis(50)
    a_op = QuantumOptics.destroy(b_motion)
    
    # Plot formatting
    colors = Dict(0=>"C0", 1=>"C1", 2=>"C2", 3=>"C3")
    linestyles = Dict(0.0=>"-", 0.2=>"--", 0.4=>":")
    
    fig = figure(figsize=(12, 8))
    println("==========================================================")
    println(" CALCULATING QCRB PHASE DIAGRAM (TROTTERIZED)")
    println(" Sweeping Channel: ", sweep_channel)
    println("==========================================================")
    
    for n in n_vals
        for r in r_vals
            println("Evaluating Fock n=$n, Squeezing r=$r ...")
            qcrb_mins = Float64[]
            
            # Prepare initial squeezed fock state
            S_op = QuantumOptics.squeeze(b_motion, r)
            psi_init = S_op * fockstate(b_motion, n)
            rho_init = dm(psi_init)
            
            for g_val in sweep_axis
                # Assign the swept value to the chosen channel
               G_phi = sweep_channel == :dephasing ? g_val : base_dephasing
                G_cool = sweep_channel == :cooling ? g_val : base_cooling
                G_heat = sweep_channel == :heating ? g_val : base_heating
                
                # INCREASE STEPS TO 500 TO PREVENT EULER DIVERGENCE AT HIGH NOISE
                safe_steps = 500
                
                # Propagate signal and perturbed signal
                rho_final = get_stepwise_combined_theory_rho(rho_init, G_phi, G_cool, G_heat, beta_signal, safe_steps)
                rho_final_pert = get_stepwise_combined_theory_rho(rho_init, G_phi, G_cool, G_heat, beta_signal + d_beta, safe_steps)
                
                # Shift back to readout frame
                D_shift = displace(b_motion, -expect(a_op, rho_final))
                rho_shift = D_shift * rho_final * dagger(D_shift)
                rho_shift_pert = D_shift * rho_final_pert * dagger(D_shift)
                
                # Tensor with spin ground state for ancilla readout
                spin_g = basisstate(b_spin, 1)
                sys_state = dm(spin_g) ⊗ rho_shift
                sys_state_pert = dm(spin_g) ⊗ rho_shift_pert
                
                # Optimize over readout alpha
                min_qcrb = Inf
                for a in alphas
                    out_state = apply_readout_SDF(sys_state, a)
                    out_state_pert = apply_readout_SDF(sys_state_pert, a)
                    
                    spin_out = ptrace(out_state, 2)
                    spin_out_pert = ptrace(out_state_pert, 2)
                    
                    qcrb = calculate_qcrb(spin_out, spin_out_pert, d_beta)
                    if qcrb < min_qcrb
                        min_qcrb = qcrb
                    end
                end
                push!(qcrb_mins, min_qcrb)
            end
            
            # Plot the resulting curve
            lbl = "n=$n, r=$r"
            plot(sweep_axis, qcrb_mins, color=colors[n], linestyle=linestyles[r], linewidth=2.5, label=lbl)
        end
    end
    
    xscale("log")
    yscale("log")
    
    xlabel("Dimensionless Noise (" * String(sweep_channel) * ")", fontsize=12)
    ylabel("min(QCRB)", fontsize=12)
    
    title_str = "QCRB Phase Diagram (Squeezed Fock States)\nSweeping: $(sweep_channel) | Base: Γ_ϕt=$(base_dephasing), Γ_ct=$(base_cooling), Γ_ht=$(base_heating)"
    title(title_str, fontsize=14)
    
    grid(alpha=0.4)
    legend(fontsize=9, loc="center left", bbox_to_anchor=(1.02, 0.5), ncol=2)
    tight_layout()
    
    display(fig)
end

# Execute the sweep on the dephasing axis
# (You can change this to :cooling or :heating to sweep those axes instead)
plot_multichannel_phase_diagram(sweep_channel=:dephasing, beta_signal=2.0)

In [ ]:
plot_multichannel_phase_diagram(sweep_channel=:cooling, beta_signal=2.0)

In [ ]:
ENV["MPLBACKEND"] = "agg"
using QuantumOptics
using PyPlot
using LinearAlgebra

# -------------------------------------------------------------
# QCRB Calculation 
# -------------------------------------------------------------
function calculate_qcrb(state, state_perturbed, d_alpha::Float64; tol=1e-10)
    rho_dense_data = dense(state).data
    rho_pert_dense_data = dense(state_perturbed).data
    drho_data = (rho_pert_dense_data - rho_dense_data) / d_alpha
    λ, V = LinearAlgebra.eigen(Hermitian(rho_dense_data))
    
    qfi = 0.0
    n = length(λ)
    for i in 1:n
        for j in 1:n
            if real(λ[i] + λ[j]) > tol
                @views v_i = V[:, i]
                @views v_j = V[:, j]
                drho_ij = dot(v_i, drho_data * v_j)
                qfi += 2 * (abs(drho_ij)^2) / real(λ[i] + λ[j])
            end
        end
    end
    return qfi > 0 ? 1.0 / qfi : Inf
end

# -------------------------------------------------------------
# Stepwise Trotter Expansion (Generalized for arbitrary initial state)
# -------------------------------------------------------------
function get_stepwise_combined_theory_rho(initial_rho::Operator, Gamma_phi_t::Float64, Gamma_cool_t::Float64, Gamma_heat_t::Float64, beta_target::Float64, steps::Int=40)
    b = initial_rho.basis_l
    N_cut = b.N
    
    a_op = QuantumOptics.destroy(b)
    at_op = QuantumOptics.create(b)
    n_op = QuantumOptics.number(b)
    I_op = QuantumOptics.identityoperator(b)
    
    d_beta = beta_target / steps
    d_g_phi = Gamma_phi_t / steps
    d_g_c = Gamma_cool_t / steps
    d_g_h = Gamma_heat_t / steps
    
    D_step = QuantumOptics.displace(b, d_beta)
    D_step_dag = QuantumOptics.dagger(D_step)
    
    M0_c = I_op - (d_g_c / 2.0) * n_op
    M1_c = sqrt(d_g_c) * a_op
    
    M0_h = I_op - (d_g_h / 2.0) * (n_op + I_op)
    M1_h = sqrt(d_g_h) * at_op
    
    rho = deepcopy(initial_rho)
    
    for k in 1:steps
        rho = D_step * rho * D_step_dag
        
        if d_g_phi > 0
            for i in 1:(N_cut + 1)
                for j in 1:(N_cut + 1)
                    rho.data[i, j] *= exp(-0.5 * d_g_phi * ((i - 1) - (j - 1))^2)
                end
            end
        end
        if d_g_c > 0
            rho = M0_c * rho * dagger(M0_c) + M1_c * rho * dagger(M1_c)
        end
        if d_g_h > 0
            rho = M0_h * rho * dagger(M0_h) + M1_h * rho * dagger(M1_h)
        end
    end
    return rho
end

# -------------------------------------------------------------
# Readout Pulse (Instantaneous Unitary SDF)
# -------------------------------------------------------------
function apply_readout_SDF(sys_rho::Operator, alpha::Float64)
    # sys_rho is assumed to be Spin ⊗ Motion
    b_spin = sys_rho.basis_l.bases[1]
    b_motion = sys_rho.basis_l.bases[2]
    
    g = basisstate(b_spin, 1)
    e = basisstate(b_spin, 2)
    x_plus = normalize(g + e)
    x_minus = normalize(g - e)
    
    P_plus = x_plus ⊗ dagger(x_plus)
    P_minus = x_minus ⊗ dagger(x_minus)
    
    # pi/2 phase shift for readout
    alpha_complex = alpha * exp(1im * pi/2)
    D_plus = QuantumOptics.displace(b_motion, alpha_complex)
    D_minus = QuantumOptics.displace(b_motion, -alpha_complex)
    
    U_SDF = (P_plus ⊗ D_plus) + (P_minus ⊗ D_minus)
    return U_SDF * sys_rho * dagger(U_SDF)
end

# -------------------------------------------------------------
# Phase Diagram Plotter (Sweeping n, r, and a target noise channel)
# -------------------------------------------------------------
function plot_multichannel_phase_diagram(; sweep_channel::Symbol=:dephasing, beta_signal::Float64=2.0)
    # Define the sweep ranges
    g_min_sweep = 1e-3
    g_max_sweep = 1e1
    sweep_axis = 10.0 .^ range(log10(g_min_sweep), log10(g_max_sweep), length=25)
    
    # Squeezing and Fock state configurations
    n_vals = [0, 1, 2, 3]
    r_vals = [0.0]
    
    # Fixed base noise (Dimensionless Γ*t)
    base_dephasing = 0.0
    base_cooling = 0.0
    base_heating = 0.0
    
    # Alphas to optimize over for QCRB
    alphas = collect(0.1:0.02:2.0)
    d_beta = 1e-4
    
    # Bases
    b_spin = SpinBasis(1//2)
    b_motion = FockBasis(50)
    a_op = QuantumOptics.destroy(b_motion)
    
    # Plot formatting
    colors = Dict(0=>"C0", 1=>"C1", 2=>"C2", 3=>"C3")
    linestyles = Dict(0.0=>"-", 0.2=>"--", 0.4=>":")
    
    fig = figure(figsize=(12, 8))
    println("==========================================================")
    println(" CALCULATING QCRB PHASE DIAGRAM (TROTTERIZED)")
    println(" Sweeping Channel: ", sweep_channel)
    println("==========================================================")
    
    for n in n_vals
        for r in r_vals
            println("Evaluating Fock n=$n, Squeezing r=$r ...")
            qcrb_mins = Float64[]
            
            # Prepare initial squeezed fock state
            S_op = QuantumOptics.squeeze(b_motion, r)
            psi_init = S_op * fockstate(b_motion, n)
            rho_init = dm(psi_init)
            
            for g_val in sweep_axis
                # Assign the swept value to the chosen channel
               G_phi = sweep_channel == :dephasing ? g_val : base_dephasing
                G_cool = sweep_channel == :cooling ? g_val : base_cooling
                G_heat = sweep_channel == :heating ? g_val : base_heating
                
                # INCREASE STEPS TO 500 TO PREVENT EULER DIVERGENCE AT HIGH NOISE
                safe_steps = 500
                
                # Propagate signal and perturbed signal
                rho_final = get_stepwise_combined_theory_rho(rho_init, G_phi, G_cool, G_heat, beta_signal, safe_steps)
                rho_final_pert = get_stepwise_combined_theory_rho(rho_init, G_phi, G_cool, G_heat, beta_signal + d_beta, safe_steps)
                
                # Shift back to readout frame
                D_shift = displace(b_motion, -expect(a_op, rho_final))
                rho_shift = D_shift * rho_final * dagger(D_shift)
                rho_shift_pert = D_shift * rho_final_pert * dagger(D_shift)
                
                # Tensor with spin ground state for ancilla readout
                spin_g = basisstate(b_spin, 1)
                sys_state = dm(spin_g) ⊗ rho_shift
                sys_state_pert = dm(spin_g) ⊗ rho_shift_pert
                
                # Optimize over readout alpha
                min_qcrb = Inf
                for a in alphas
                    out_state = apply_readout_SDF(sys_state, a)
                    out_state_pert = apply_readout_SDF(sys_state_pert, a)
                    
                    spin_out = ptrace(out_state, 2)
                    spin_out_pert = ptrace(out_state_pert, 2)
                    
                    qcrb = calculate_qcrb(spin_out, spin_out_pert, d_beta)
                    if qcrb < min_qcrb
                        min_qcrb = qcrb
                    end
                end
                push!(qcrb_mins, min_qcrb)
            end
            
            # Plot the resulting curve
            lbl = "n=$n, r=$r"
            plot(sweep_axis, qcrb_mins, color=colors[n], linestyle=linestyles[r], linewidth=2.5, label=lbl)
        end
    end
    
    xscale("log")
    yscale("log")
    
    xlabel("Dimensionless Noise (" * String(sweep_channel) * ")", fontsize=12)
    ylabel("min(QCRB)", fontsize=12)
    
    title_str = "QCRB Phase Diagram (Squeezed Fock States)\nSweeping: $(sweep_channel) | Base: Γ_ϕt=$(base_dephasing), Γ_ct=$(base_cooling), Γ_ht=$(base_heating)"
    title(title_str, fontsize=14)
    
    grid(alpha=0.4)
    legend(fontsize=9, loc="center left", bbox_to_anchor=(1.02, 0.5), ncol=2)
    tight_layout()
    
    display(fig)
end

# Execute the sweep on the dephasing axis
# (You can change this to :cooling or :heating to sweep those axes instead)
plot_multichannel_phase_diagram(sweep_channel=:cooling, beta_signal=2.0)

In [ ]:
ENV["MPLBACKEND"] = "agg"
using QuantumOptics
using PyPlot
using LinearAlgebra

# -------------------------------------------------------------
# QCRB Calculation 
# -------------------------------------------------------------
function calculate_qcrb(state, state_perturbed, d_alpha::Float64; tol=1e-10)
    rho_dense_data = dense(state).data
    rho_pert_dense_data = dense(state_perturbed).data
    drho_data = (rho_pert_dense_data - rho_dense_data) / d_alpha
    λ, V = LinearAlgebra.eigen(Hermitian(rho_dense_data))
    
    qfi = 0.0
    n = length(λ)
    for i in 1:n
        for j in 1:n
            if real(λ[i] + λ[j]) > tol
                @views v_i = V[:, i]
                @views v_j = V[:, j]
                drho_ij = dot(v_i, drho_data * v_j)
                qfi += 2 * (abs(drho_ij)^2) / real(λ[i] + λ[j])
            end
        end
    end
    return qfi > 0 ? 1.0 / qfi : Inf
end

# -------------------------------------------------------------
# Stepwise Trotter Expansion (Generalized for arbitrary initial state)
# -------------------------------------------------------------
function get_stepwise_combined_theory_rho(initial_rho::Operator, Gamma_phi_t::Float64, Gamma_cool_t::Float64, Gamma_heat_t::Float64, beta_target::Float64, steps::Int=40)
    b = initial_rho.basis_l
    N_cut = b.N
    
    a_op = QuantumOptics.destroy(b)
    at_op = QuantumOptics.create(b)
    n_op = QuantumOptics.number(b)
    I_op = QuantumOptics.identityoperator(b)
    
    d_beta = beta_target / steps
    d_g_phi = Gamma_phi_t / steps
    d_g_c = Gamma_cool_t / steps
    d_g_h = Gamma_heat_t / steps
    
    D_step = QuantumOptics.displace(b, d_beta)
    D_step_dag = QuantumOptics.dagger(D_step)
    
    M0_c = I_op - (d_g_c / 2.0) * n_op
    M1_c = sqrt(d_g_c) * a_op
    
    M0_h = I_op - (d_g_h / 2.0) * (n_op + I_op)
    M1_h = sqrt(d_g_h) * at_op
    
    rho = deepcopy(initial_rho)
    
    for k in 1:steps
        rho = D_step * rho * D_step_dag
        
        if d_g_phi > 0
            for i in 1:(N_cut + 1)
                for j in 1:(N_cut + 1)
                    rho.data[i, j] *= exp(-0.5 * d_g_phi * ((i - 1) - (j - 1))^2)
                end
            end
        end
        if d_g_c > 0
            rho = M0_c * rho * dagger(M0_c) + M1_c * rho * dagger(M1_c)
        end
        if d_g_h > 0
            rho = M0_h * rho * dagger(M0_h) + M1_h * rho * dagger(M1_h)
        end
    end
    return rho
end

# -------------------------------------------------------------
# Readout Pulse (Instantaneous Unitary SDF)
# -------------------------------------------------------------
function apply_readout_SDF(sys_rho::Operator, alpha::Float64)
    # sys_rho is assumed to be Spin ⊗ Motion
    b_spin = sys_rho.basis_l.bases[1]
    b_motion = sys_rho.basis_l.bases[2]
    
    g = basisstate(b_spin, 1)
    e = basisstate(b_spin, 2)
    x_plus = normalize(g + e)
    x_minus = normalize(g - e)
    
    P_plus = x_plus ⊗ dagger(x_plus)
    P_minus = x_minus ⊗ dagger(x_minus)
    
    # pi/2 phase shift for readout
    alpha_complex = alpha * exp(1im * pi/2)
    D_plus = QuantumOptics.displace(b_motion, alpha_complex)
    D_minus = QuantumOptics.displace(b_motion, -alpha_complex)
    
    U_SDF = (P_plus ⊗ D_plus) + (P_minus ⊗ D_minus)
    return U_SDF * sys_rho * dagger(U_SDF)
end

# -------------------------------------------------------------
# Phase Diagram Plotter (Sweeping n, r, and a target noise channel)
# -------------------------------------------------------------
function plot_multichannel_phase_diagram(; sweep_channel::Symbol=:dephasing, beta_signal::Float64=2.0)
    # Define the sweep ranges
    g_min_sweep = 1e-3
    g_max_sweep = 1e1
    sweep_axis = 10.0 .^ range(log10(g_min_sweep), log10(g_max_sweep), length=50)
    
    # Squeezing and Fock state configurations
    n_vals = [0, 1, 2, 3]
    r_vals = [0.0]
    
    # Fixed base noise (Dimensionless Γ*t)
    base_dephasing = 0.0
    base_cooling = 0.0
    base_heating = 0.0
    
    # Alphas to optimize over for QCRB
    alphas = collect(0.1:0.01:2.0)
    d_beta = 1e-4
    
    # Bases
    b_spin = SpinBasis(1//2)
    b_motion = FockBasis(50)
    a_op = QuantumOptics.destroy(b_motion)
    
    # Plot formatting
    colors = Dict(0=>"C0", 1=>"C1", 2=>"C2", 3=>"C3")
    linestyles = Dict(0.0=>"-", 0.2=>"--", 0.4=>":")
    
    fig = figure(figsize=(12, 8))
    println("==========================================================")
    println(" CALCULATING QCRB PHASE DIAGRAM (TROTTERIZED)")
    println(" Sweeping Channel: ", sweep_channel)
    println("==========================================================")
    
    for n in n_vals
        for r in r_vals
            println("Evaluating Fock n=$n, Squeezing r=$r ...")
            qcrb_mins = Float64[]
            
            # Prepare initial squeezed fock state
            S_op = QuantumOptics.squeeze(b_motion, r)
            psi_init = S_op * fockstate(b_motion, n)
            rho_init = dm(psi_init)
            
            for g_val in sweep_axis
                # Assign the swept value to the chosen channel
               G_phi = sweep_channel == :dephasing ? g_val : base_dephasing
                G_cool = sweep_channel == :cooling ? g_val : base_cooling
                G_heat = sweep_channel == :heating ? g_val : base_heating
                
                # INCREASE STEPS TO 500 TO PREVENT EULER DIVERGENCE AT HIGH NOISE
                safe_steps = 500
                
                # Propagate signal and perturbed signal
                rho_final = get_stepwise_combined_theory_rho(rho_init, G_phi, G_cool, G_heat, beta_signal, safe_steps)
                rho_final_pert = get_stepwise_combined_theory_rho(rho_init, G_phi, G_cool, G_heat, beta_signal + d_beta, safe_steps)
                
                # Shift back to readout frame
                D_shift = displace(b_motion, -expect(a_op, rho_final))
                rho_shift = D_shift * rho_final * dagger(D_shift)
                rho_shift_pert = D_shift * rho_final_pert * dagger(D_shift)
                
                # Tensor with spin ground state for ancilla readout
                spin_g = basisstate(b_spin, 1)
                sys_state = dm(spin_g) ⊗ rho_shift
                sys_state_pert = dm(spin_g) ⊗ rho_shift_pert
                
                # Optimize over readout alpha
                min_qcrb = Inf
                for a in alphas
                    out_state = apply_readout_SDF(sys_state, a)
                    out_state_pert = apply_readout_SDF(sys_state_pert, a)
                    
                    spin_out = ptrace(out_state, 2)
                    spin_out_pert = ptrace(out_state_pert, 2)
                    
                    qcrb = calculate_qcrb(spin_out, spin_out_pert, d_beta)
                    if qcrb < min_qcrb
                        min_qcrb = qcrb
                    end
                end
                push!(qcrb_mins, min_qcrb)
            end
            
            # Plot the resulting curve
            lbl = "n=$n, r=$r"
            plot(sweep_axis, qcrb_mins, color=colors[n], linestyle=linestyles[r], linewidth=2.5, label=lbl)
        end
    end
    
    xscale("log")
    yscale("log")
    
    xlabel("Dimensionless Noise (" * String(sweep_channel) * ")", fontsize=12)
    ylabel("min(QCRB)", fontsize=12)
    
    title_str = "QCRB Phase Diagram (Squeezed Fock States)\nSweeping: $(sweep_channel) | Base: Γ_ϕt=$(base_dephasing), Γ_ct=$(base_cooling), Γ_ht=$(base_heating)"
    title(title_str, fontsize=14)
    
    grid(alpha=0.4)
    legend(fontsize=9, loc="center left", bbox_to_anchor=(1.02, 0.5), ncol=2)
    tight_layout()
    
    display(fig)
end

# change this to :cooling or :heating to sweep those axes instead
plot_multichannel_phase_diagram(sweep_channel=:cooling, beta_signal=2.0)

In [ ]:
ENV["MPLBACKEND"] = "agg"
using QuantumOptics
using PyPlot
using LinearAlgebra

# -------------------------------------------------------------
# QCRB Calculation 
# -------------------------------------------------------------
function calculate_qcrb(state, state_perturbed, d_alpha::Float64; tol=1e-10)
    rho_dense_data = dense(state).data
    rho_pert_dense_data = dense(state_perturbed).data
    drho_data = (rho_pert_dense_data - rho_dense_data) / d_alpha
    λ, V = LinearAlgebra.eigen(Hermitian(rho_dense_data))
    
    qfi = 0.0
    n = length(λ)
    for i in 1:n
        for j in 1:n
            if real(λ[i] + λ[j]) > tol
                @views v_i = V[:, i]
                @views v_j = V[:, j]
                drho_ij = dot(v_i, drho_data * v_j)
                qfi += 2 * (abs(drho_ij)^2) / real(λ[i] + λ[j])
            end
        end
    end
    return qfi > 0 ? 1.0 / qfi : Inf
end

# -------------------------------------------------------------
# Stepwise Trotter Expansion
# -------------------------------------------------------------
function get_stepwise_combined_theory_rho(initial_rho::Operator, Gamma_phi_t::Float64, Gamma_cool_t::Float64, Gamma_heat_t::Float64, beta_target::Float64, steps::Int=40)
    b = initial_rho.basis_l
    N_cut = b.N
    
    a_op = QuantumOptics.destroy(b)
    at_op = QuantumOptics.create(b)
    n_op = QuantumOptics.number(b)
    I_op = QuantumOptics.identityoperator(b)
    
    d_beta = beta_target / steps
    d_g_phi = Gamma_phi_t / steps
    d_g_c = Gamma_cool_t / steps
    d_g_h = Gamma_heat_t / steps
    
    D_step = QuantumOptics.displace(b, d_beta)
    D_step_dag = QuantumOptics.dagger(D_step)
    
    M0_c = I_op - (d_g_c / 2.0) * n_op
    M1_c = sqrt(d_g_c) * a_op
    
    M0_h = I_op - (d_g_h / 2.0) * (n_op + I_op)
    M1_h = sqrt(d_g_h) * at_op
    
    rho = deepcopy(initial_rho)
    
    for k in 1:steps
        rho = D_step * rho * D_step_dag
        
        if d_g_phi > 0
            for i in 1:(N_cut + 1)
                for j in 1:(N_cut + 1)
                    rho.data[i, j] *= exp(-0.5 * d_g_phi * ((i - 1) - (j - 1))^2)
                end
            end
        end
        if d_g_c > 0
            rho = M0_c * rho * dagger(M0_c) + M1_c * rho * dagger(M1_c)
        end
        if d_g_h > 0
            rho = M0_h * rho * dagger(M0_h) + M1_h * rho * dagger(M1_h)
        end
    end
    return rho
end

# -------------------------------------------------------------
# Readout Pulse
# -------------------------------------------------------------
function apply_readout_SDF(sys_rho::Operator, alpha::Float64)
    b_spin = sys_rho.basis_l.bases[1]
    b_motion = sys_rho.basis_l.bases[2]
    
    g = basisstate(b_spin, 1)
    e = basisstate(b_spin, 2)
    x_plus = normalize(g + e)
    x_minus = normalize(g - e)
    
    P_plus = x_plus ⊗ dagger(x_plus)
    P_minus = x_minus ⊗ dagger(x_minus)
    
    alpha_complex = alpha * exp(1im * pi/2)
    D_plus = QuantumOptics.displace(b_motion, alpha_complex)
    D_minus = QuantumOptics.displace(b_motion, -alpha_complex)
    
    U_SDF = (P_plus ⊗ D_plus) + (P_minus ⊗ D_minus)
    return U_SDF * sys_rho * dagger(U_SDF)
end

# -------------------------------------------------------------
# Phase Diagram Plotter (With Crossover Extraction)
# -------------------------------------------------------------
function plot_multichannel_phase_diagram(; sweep_channel::Symbol=:dephasing, beta_signal::Float64=2.0)
    g_min_sweep = 1e-2
    g_max_sweep = 1e2
    sweep_axis = 10.0 .^ range(log10(g_min_sweep), log10(g_max_sweep), length=100)
    
    n_vals = [0, 1, 2, 3]
    r_vals = [0.0]
    
    base_dephasing = 0.0
    base_cooling = 0.0
    base_heating = 0.0
    
    alphas = collect(0.1:0.01:2.0)
    d_beta = 1e-4
    
    b_spin = SpinBasis(1//2)
    b_motion = FockBasis(50)
    a_op = QuantumOptics.destroy(b_motion)
    
    colors = Dict(0=>"C0", 1=>"C1", 2=>"C2", 3=>"C3")
    linestyles = Dict(0.0=>"-", 0.2=>"--", 0.4=>":")
    
    # DICTIONARY TO STORE CURVES FOR CROSSOVER COMPARISON
    curve_data = Dict()
    
    fig = figure(figsize=(12, 8))
    println("==========================================================")
    println(" CALCULATING QCRB PHASE DIAGRAM (TROTTERIZED)")
    println("==========================================================")
    
    for n in n_vals
        for r in r_vals
            println("Evaluating Fock n=$n, Squeezing r=$r ...")
            qcrb_mins = Float64[]
            
            S_op = QuantumOptics.squeeze(b_motion, r)
            psi_init = S_op * fockstate(b_motion, n)
            rho_init = dm(psi_init)
            
            for g_val in sweep_axis
                G_phi = sweep_channel == :dephasing ? g_val : base_dephasing
                G_cool = sweep_channel == :cooling ? g_val : base_cooling
                G_heat = sweep_channel == :heating ? g_val : base_heating
                
                # DYNAMIC STEP COUNT: Prevents Euler divergence at high noise
                safe_steps = max(500, Int(ceil(4.0 * g_val * b_motion.N)))
                
                rho_final = get_stepwise_combined_theory_rho(rho_init, G_phi, G_cool, G_heat, beta_signal, safe_steps)
                rho_final_pert = get_stepwise_combined_theory_rho(rho_init, G_phi, G_cool, G_heat, beta_signal + d_beta, safe_steps)
                
                D_shift = displace(b_motion, -expect(a_op, rho_final))
                rho_shift = D_shift * rho_final * dagger(D_shift)
                rho_shift_pert = D_shift * rho_final_pert * dagger(D_shift)
                
                spin_g = basisstate(b_spin, 1)
                sys_state = dm(spin_g) ⊗ rho_shift
                sys_state_pert = dm(spin_g) ⊗ rho_shift_pert
                
                min_qcrb = Inf
                for a in alphas
                    out_state = apply_readout_SDF(sys_state, a)
                    out_state_pert = apply_readout_SDF(sys_state_pert, a)
                    
                    spin_out = ptrace(out_state, 2)
                    spin_out_pert = ptrace(out_state_pert, 2)
                    
                    qcrb = calculate_qcrb(spin_out, spin_out_pert, d_beta)
                    if qcrb < min_qcrb
                        min_qcrb = qcrb
                    end
                end
                push!(qcrb_mins, min_qcrb)
            end
            
            # Store the data for intersection logic
            curve_data[(n, r)] = qcrb_mins
            lbl = "n=$n, r=$r"
            plot(sweep_axis, qcrb_mins, color=colors[n], linestyle=linestyles[r], linewidth=2.5, label=lbl)
        end
    end
    
    # -------------------------------------------------------------
    # EXTRACT CROSSOVERS
    # -------------------------------------------------------------
    println("\nExtracting Crossovers...")
    cross_pts_x = Float64[]
    cross_pts_y = Float64[]
    
    for r in r_vals
        # Compare n=3 with n=2, n=2 with n=1, etc.
        for n in 1:maximum(n_vals)
            if haskey(curve_data, (n, r)) && haskey(curve_data, (n-1, r))
                y1 = curve_data[(n, r)]
                y2 = curve_data[(n-1, r)]
                
                for k in 1:(length(sweep_axis)-1)
                    diff1 = y1[k] - y2[k]
                    diff2 = y1[k+1] - y2[k+1]
                    
                    # If the difference changes sign, a crossover occurred between k and k+1
                    if diff1 * diff2 <= 0.0
                        # Log-Log Linear Interpolation for exact coordinate
                        log_x1, log_x2 = log10(sweep_axis[k]), log10(sweep_axis[k+1])
                        log_y1_A, log_y1_B = log10(y1[k]), log10(y1[k+1])
                        log_y2_A, log_y2_B = log10(y2[k]), log10(y2[k+1])
                        
                        t = (log_y2_A - log_y1_A) / (log_y1_B - log_y1_A - log_y2_B + log_y2_A)
                        
                        cross_x = 10.0^((1-t)*log_x1 + t*log_x2)
                        cross_y = 10.0^((1-t)*log_y1_A + t*log_y1_B)
                        
                        push!(cross_pts_x, cross_x)
                        push!(cross_pts_y, cross_y)
                        
                        println("Crossover (n=$(n) -> n=$(n-1), r=$r) at Dimensionless Noise = $(round(cross_x, digits=4))")
                    end
                end
            end
        end
    end
    
    # Plot the crossover markers if any were found
    if length(cross_pts_x) > 0
        scatter(cross_pts_x, cross_pts_y, color="black", s=100, zorder=5, marker="x", label="Crossovers")
    end
    
    xscale("log"); yscale("log")
    xlim(g_min_sweep, g_max_sweep)
    ylim(1e-2, 1e2)
    
    xlabel("Dimensionless Noise (" * String(sweep_channel) * ")", fontsize=12)
    ylabel("min(QCRB)", fontsize=12)
    
    title_str = "QCRB Phase Diagram (Squeezed Fock States)\nSweeping: $(sweep_channel) | Base: Γ_ϕt=$(base_dephasing), Γ_ct=$(base_cooling), Γ_ht=$(base_heating)"
    title(title_str, fontsize=14)
    
    grid(alpha=0.4)
    legend(fontsize=9, loc="center left", bbox_to_anchor=(1.02, 0.5), ncol=2)
    tight_layout()
    
    display(fig)
end

plot_multichannel_phase_diagram(sweep_channel=:cooling, beta_signal=2.0)

In [ ]:
ENV["MPLBACKEND"] = "agg"
using QuantumOptics
using PyPlot
using LinearAlgebra

# -------------------------------------------------------------
# QCRB Calculation (Overloaded for raw 2x2 Julia Matrices)
# -------------------------------------------------------------
function calculate_qcrb_raw(rho_dense_data::Matrix{ComplexF64}, rho_pert_dense_data::Matrix{ComplexF64}, d_alpha::Float64; tol=1e-10)
    drho_data = (rho_pert_dense_data - rho_dense_data) / d_alpha
    λ, V = LinearAlgebra.eigen(Hermitian(rho_dense_data))
    
    qfi = 0.0
    n = length(λ)
    for i in 1:n
        for j in 1:n
            if real(λ[i] + λ[j]) > tol
                @views v_i = V[:, i]
                @views v_j = V[:, j]
                drho_ij = dot(v_i, drho_data * v_j)
                qfi += 2 * (abs(drho_ij)^2) / real(λ[i] + λ[j])
            end
        end
    end
    return qfi > 0 ? 1.0 / qfi : Inf
end

# Fast matrix trace without allocations (relaxed to AbstractMatrix)
function tr_prod(A::AbstractMatrix, B::AbstractMatrix)
    res = zero(ComplexF64)
    @inbounds for j in axes(A, 2)
        for i in axes(A, 1)
            res += A[j, i] * B[i, j]
        end
    end
    return res
end

# -------------------------------------------------------------
# Stepwise Trotter Expansion (In-Place Memory Optimized)
# -------------------------------------------------------------
function get_stepwise_combined_theory_rho_fast!(
    rho::Matrix{ComplexF64}, tmp1::Matrix{ComplexF64}, tmp2::Matrix{ComplexF64}, tmp3::Matrix{ComplexF64},
    D_mat, D_dag_mat, M0c_mat, M0c_dag_mat, M1c_mat, M1c_dag_mat, M0h_mat, M0h_dag_mat, M1h_mat, M1h_dag_mat,
    dephasing_mask::Matrix{Float64}, d_g_phi::Float64, d_g_c::Float64, d_g_h::Float64, steps::Int)
    
    for k in 1:steps
        # Apply coherent drive: rho = D * rho * D_dag
        mul!(tmp1, D_mat, rho)
        mul!(tmp2, tmp1, D_dag_mat)
        rho .= tmp2
        
        # Apply pure dephasing
        if d_g_phi > 0
            rho .*= dephasing_mask 
        end
        
        # Apply cooling dissipator
        if d_g_c > 0
            mul!(tmp1, M0c_mat, rho)
            mul!(tmp2, tmp1, M0c_dag_mat)
            mul!(tmp1, M1c_mat, rho)
            mul!(tmp3, tmp1, M1c_dag_mat)
            rho .= tmp2 .+ tmp3
        end
        
        # Apply heating dissipator
        if d_g_h > 0
            mul!(tmp1, M0h_mat, rho)
            mul!(tmp2, tmp1, M0h_dag_mat)
            mul!(tmp1, M1h_mat, rho)
            mul!(tmp3, tmp1, M1h_dag_mat)
            rho .= tmp2 .+ tmp3
        end
    end
    return rho
end

# -------------------------------------------------------------
# Phase Diagram Plotter 
# -------------------------------------------------------------
function plot_multichannel_phase_diagram(; sweep_channel::Symbol=:dephasing, beta_signal::Float64=2.0)
    g_min_sweep = 1e-2
    g_max_sweep = 1e2
    sweep_axis = 10.0 .^ range(log10(g_min_sweep), log10(g_max_sweep), length=100)
    
    n_vals = [0, 1, 2, 3]
    r_vals = [0.0]
    
    base_dephasing = 0.0
    base_cooling = 0.0
    base_heating = 0.0
    
    alphas = collect(0.05:0.01:2.0)
    d_beta = 1e-4
    
    b_motion = FockBasis(50)
    
    # Ensure operators are dense for fast matrix multiplications
    a_op = dense(QuantumOptics.destroy(b_motion))
    at_op = dense(QuantumOptics.create(b_motion))
    n_op = dense(QuantumOptics.number(b_motion))
    I_op = dense(QuantumOptics.identityoperator(b_motion))
    
    colors = Dict(0=>"C0", 1=>"C1", 2=>"C2", 3=>"C3")
    linestyles = Dict(0.0=>"-", 0.2=>"--", 0.4=>":")
    curve_data = Dict()
    
    fig = figure(figsize=(12, 8))
    println("==========================================================")
    println(" CALCULATING QCRB PHASE DIAGRAM (ULTRA-FAST TROTTER)")
    println("==========================================================")
    
    # Precompute Readout Matrices V = D(2i*alpha)
    V_ops = [QuantumOptics.displace(b_motion, 2im * a).data for a in alphas]
    
    # Memory buffers for Trotter loop (prevents allocation thrashing)
    N_cut = b_motion.N
    tmp1 = zeros(ComplexF64, N_cut+1, N_cut+1)
    tmp2 = zeros(ComplexF64, N_cut+1, N_cut+1)
    tmp3 = zeros(ComplexF64, N_cut+1, N_cut+1)
    dephasing_mask = ones(Float64, N_cut+1, N_cut+1)
    
    for n in n_vals
        for r in r_vals
            println("Evaluating Fock n=$n, Squeezing r=$r ...")
            qcrb_mins = Float64[]
            
            S_op = QuantumOptics.squeeze(b_motion, r)
            psi_init = S_op * fockstate(b_motion, n)
            rho_init_data = dm(psi_init).data
            
            for g_val in sweep_axis
                G_phi = sweep_channel == :dephasing ? g_val : base_dephasing
                G_cool = sweep_channel == :cooling ? g_val : base_cooling
                G_heat = sweep_channel == :heating ? g_val : base_heating
                
                # LOWERED CEILING: Speeds up execution without crashing
                safe_steps = max(500, Int(ceil(0.6 * g_val * b_motion.N)))
                
                d_beta_step = beta_signal / safe_steps
                d_beta_pert_step = (beta_signal + d_beta) / safe_steps
                d_g_phi = G_phi / safe_steps
                d_g_c = G_cool / safe_steps
                d_g_h = G_heat / safe_steps
                
                # Precompute Step Operators for this specific noise parameter
                D_mat = QuantumOptics.displace(b_motion, d_beta_step).data
                D_pert_mat = QuantumOptics.displace(b_motion, d_beta_pert_step).data
                
                M0c_mat = (I_op - (d_g_c / 2.0) * n_op).data
                M1c_mat = (sqrt(d_g_c) * a_op).data
                M0h_mat = (I_op - (d_g_h / 2.0) * (n_op + I_op)).data
                M1h_mat = (sqrt(d_g_h) * at_op).data
                
                # Update dephasing mask
                if d_g_phi > 0
                    for i in 1:(N_cut + 1), j in 1:(N_cut + 1)
                        dephasing_mask[i, j] = exp(-0.5 * d_g_phi * ((i - 1) - (j - 1))^2)
                    end
                end
                
                # Propagate Signal
                rho_final = get_stepwise_combined_theory_rho_fast!(copy(rho_init_data), tmp1, tmp2, tmp3,
                    D_mat, D_mat', M0c_mat, M0c_mat', M1c_mat, M1c_mat', M0h_mat, M0h_mat', M1h_mat, M1h_mat',
                    dephasing_mask, d_g_phi, d_g_c, d_g_h, safe_steps)
                
                # Force Hermiticity and Trace Normalization to prevent Trotter runaway
                rho_final .= (rho_final .+ rho_final') ./ 2.0
                rho_final ./= tr(rho_final)
                    
                # Propagate Perturbed Signal
                rho_final_pert = get_stepwise_combined_theory_rho_fast!(copy(rho_init_data), tmp1, tmp2, tmp3,
                    D_pert_mat, D_pert_mat', M0c_mat, M0c_mat', M1c_mat, M1c_mat', M0h_mat, M0h_mat', M1h_mat, M1h_mat',
                    dephasing_mask, d_g_phi, d_g_c, d_g_h, safe_steps)
                
                # Force Hermiticity and Trace Normalization
                rho_final_pert .= (rho_final_pert .+ rho_final_pert') ./ 2.0
                rho_final_pert ./= tr(rho_final_pert)
                
                # Shift back to readout frame
                shift_val = -tr_prod(a_op.data, rho_final)
                shift_pert_val = -tr_prod(a_op.data, rho_final_pert)
                
                D_shift = QuantumOptics.displace(b_motion, shift_val).data
                D_shift_pert = QuantumOptics.displace(b_motion, shift_pert_val).data
                
                rho_shift = D_shift * rho_final * D_shift'
                rho_shift_pert = D_shift_pert * rho_final_pert * D_shift_pert'
                
                # Analytic Readout Evaluation (Bypassing Tensor Products)
                min_qcrb = Inf
                for (idx, a) in enumerate(alphas)
                    V_mat = V_ops[idx]
                    v_val = tr_prod(V_mat, rho_shift)
                    v_val_pert = tr_prod(V_mat, rho_shift_pert)
                    
                    spin_out = ComplexF64[0.5  0.5*v_val; 0.5*conj(v_val)  0.5]
                    spin_out_pert = ComplexF64[0.5  0.5*v_val_pert; 0.5*conj(v_val_pert)  0.5]
                    
                    qcrb = calculate_qcrb_raw(spin_out, spin_out_pert, d_beta)
                    if qcrb < min_qcrb
                        min_qcrb = qcrb
                    end
                end
                push!(qcrb_mins, min_qcrb)
            end
            
            curve_data[(n, r)] = qcrb_mins
            lbl = "n=$n, r=$r"
            plot(sweep_axis, qcrb_mins, color=colors[n], linestyle=linestyles[r], linewidth=2.5, label=lbl)
        end
    end
    
    # -------------------------------------------------------------
    # EXTRACT CROSSOVERS
    # -------------------------------------------------------------
    println("\nExtracting Crossovers...")
    cross_pts_x = Float64[]
    cross_pts_y = Float64[]
    
    for r in r_vals
        for n in 1:maximum(n_vals)
            if haskey(curve_data, (n, r)) && haskey(curve_data, (n-1, r))
                y1 = curve_data[(n, r)]
                y2 = curve_data[(n-1, r)]
                for k in 1:(length(sweep_axis)-1)
                    if (y1[k] - y2[k]) * (y1[k+1] - y2[k+1]) <= 0.0
                        log_x1, log_x2 = log10(sweep_axis[k]), log10(sweep_axis[k+1])
                        log_y1_A, log_y1_B = log10(y1[k]), log10(y1[k+1])
                        log_y2_A, log_y2_B = log10(y2[k]), log10(y2[k+1])
                        
                        t = (log_y2_A - log_y1_A) / (log_y1_B - log_y1_A - log_y2_B + log_y2_A)
                        cross_x = 10.0^((1-t)*log_x1 + t*log_x2)
                        cross_y = 10.0^((1-t)*log_y1_A + t*log_y1_B)
                        
                        push!(cross_pts_x, cross_x)
                        push!(cross_pts_y, cross_y)
                        println("Crossover (n=$(n) -> n=$(n-1), r=$r) at Dimensionless Noise = $(round(cross_x, digits=4))")
                        
                        break # Stops after finding the first crossing to ignore thermal wall errors
                    end
                end
            end
        end
    end
    
    if length(cross_pts_x) > 0
        scatter(cross_pts_x, cross_pts_y, color="black", s=100, zorder=5, marker="x", label="Crossovers")
    end
    
    xscale("log"); yscale("log")
    xlim(g_min_sweep, g_max_sweep)
    ylim(1e-2, 1e2)
    
    xlabel("Dimensionless Noise (" * String(sweep_channel) * ")", fontsize=12)
    ylabel("min(QCRB)", fontsize=12)
    title("QCRB Phase Diagram (Squeezed Fock States)\nSweeping: $(sweep_channel) | Base: Γ_ϕt=$(base_dephasing), Γ_ct=$(base_cooling), Γ_ht=$(base_heating)", fontsize=14)
    grid(alpha=0.4)
    legend(fontsize=9, loc="center left", bbox_to_anchor=(1.02, 0.5), ncol=2)
    tight_layout()
    display(fig)
end

plot_multichannel_phase_diagram(sweep_channel=:cooling, beta_signal=2.0)